In [1]:
# Cell 1: نصب پکیج‌های مورد نیاز
!pip install -q transformer-lens torch numpy accelerate

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 30.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.6/56.6 kB 6.0 MB/s eta 0:00:00


In [2]:
"""
m7_10_adaptive_self_probing.py
===============================
Milestone 7.10: Adaptive Micro-Probing vs. Fixed Probe Sequences.

Core Objective:
Validate if an active sequential policy can achieve Full-Suite accuracy (K=4)
using an adaptive budget (K <= 2) via Bayesian Information Gain.

Ablation Target: L21H06 Zero-Ablation (Target Damage)
Probe Library:
  - P1: Scale-Down (z * 0.90)
  - P2: Scale-Up (z * 1.10)
  - P3: Orthogonal Jitter (z + 0.05 * v_ortho)
  - P4: Directional Damping (z * 0.92)
"""

import torch
import numpy as np
import json
from scipy.stats import spearmanr
from typing import Dict, List, Tuple
from transformer_lens import HookedTransformer

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)


# ==============================================================================
# 1. MECHANISTIC PROBING HARNESS
# ==============================================================================

class M7_10_Harness:
    def __init__(self, model_name: str = "Qwen/Qwen2.5-0.5B", device: str = "cuda"):
        self.device = device if torch.cuda.is_available() else "cpu"
        print(f"🔄 Loading {model_name} on {self.device.upper()} (bfloat16)...")
        self.model = HookedTransformer.from_pretrained(
            model_name,
            device=self.device,
            torch_dtype=torch.bfloat16,
            default_padding_side="left"
        )
        self.model.eval()
        self.target_layer = 21
        self.target_head = 6
        self.d_head = self.model.cfg.d_head

        # Orthogonal reference vector
        v_rand = torch.randn(self.d_head, dtype=torch.float32)
        self.v_ortho_base = (v_rand / v_rand.norm()).to(self.device)

    def compute_diff(self, logits: torch.Tensor, c_id: int, inc_id: int) -> float:
        return float((logits[0, -1, c_id] - logits[0, -1, inc_id]).detach().cpu().item())

    @torch.no_grad()
    def get_clean_and_target(self, task: Dict) -> Tuple[float, float, torch.Tensor]:
        clean_toks = self.model.to_tokens(task["clean"])
        c_id = int(self.model.to_single_token(task["correct"]))
        inc_id = int(self.model.to_single_token(task["incorrect"]))

        nom_logits, cache = self.model.run_with_cache(clean_toks)
        nom_diff = self.compute_diff(nom_logits, c_id, inc_id)
        z_orig = cache[f"blocks.{self.target_layer}.attn.hook_z"][:, -1, self.target_head, :].clone()

        # Target Zero-Ablation (Ground Truth)
        def target_hook(act, hook):
            act[:, -1, self.target_head, :] = 0.0
            return act

        with self.model.hooks(fwd_hooks=[(f"blocks.{self.target_layer}.attn.hook_z", target_hook)]):
            target_logits, _ = self.model.run_with_cache(clean_toks)
        target_damage = nom_diff - self.compute_diff(target_logits, c_id, inc_id)

        return nom_diff, target_damage, z_orig

    @torch.no_grad()
    def execute_single_probe(self, task: Dict, probe_id: int, nom_diff: float, z_orig: torch.Tensor) -> float:
        clean_toks = self.model.to_tokens(task["clean"])
        c_id = int(self.model.to_single_token(task["correct"]))
        inc_id = int(self.model.to_single_token(task["incorrect"]))

        if probe_id == 0:  # P1: Scale Down 0.90
            def p_hook(act, hook):
                act[:, -1, self.target_head, :] = act[:, -1, self.target_head, :] * 0.90
                return act
        elif probe_id == 1:  # P2: Scale Up 1.10
            def p_hook(act, hook):
                act[:, -1, self.target_head, :] = act[:, -1, self.target_head, :] * 1.10
                return act
        elif probe_id == 2:  # P3: Orthogonal Jitter
            z_n = z_orig[0] / (z_orig[0].norm() + 1e-6)
            v_proj = torch.dot(self.v_ortho_base.to(z_n.dtype), z_n) * z_n
            v_ortho = self.v_ortho_base.to(z_n.dtype) - v_proj
            v_ortho = v_ortho / (v_ortho.norm() + 1e-6)
            def p_hook(act, hook):
                act[:, -1, self.target_head, :] = act[:, -1, self.target_head, :] + 0.05 * act[:, -1, self.target_head, :].norm() * v_ortho
                return act
        elif probe_id == 3:  # P4: Damping 0.92
            def p_hook(act, hook):
                act[:, -1, self.target_head, :] = act[:, -1, self.target_head, :] * 0.92
                return act
        else:
            raise ValueError(f"Unknown probe_id: {probe_id}")

        with self.model.hooks(fwd_hooks=[(f"blocks.{self.target_layer}.attn.hook_z", p_hook)]):
            p_logits, _ = self.model.run_with_cache(clean_toks)

        probe_response = nom_diff - self.compute_diff(p_logits, c_id, inc_id)
        return float(probe_response)


# ==============================================================================
# 2. BAYESIAN SELF-MODEL & ADAPTIVE PROBE POLICY
# ==============================================================================

class BayesianProbeSelfModel:
    """Maintains a joint Gaussian distribution over probe responses and target damage."""
    def __init__(self, train_data: List[Dict]):
        # X: (N, 4) probe responses, y: (N,) target damage
        self.X_train = np.array([r["probes"] for r in train_data])
        self.y_train = np.array([r["target"] for r in train_data])

        # Empirical joint covariance matrix Sigma of shape (5, 5) [P1, P2, P3, P4, Target]
        joint_data = np.hstack([self.X_train, self.y_train.reshape(-1, 1)])
        self.mu_joint = np.mean(joint_data, axis=0)
        self.sigma_joint = np.cov(joint_data, rowvar=False) + 1e-4 * np.eye(5)

    def get_initial_uncertainty(self) -> float:
        return float(self.sigma_joint[4, 4])

    def predict_given_observed(self, observed_dict: Dict[int, float]) -> Tuple[float, float]:
        """Gaussian conditioning on subset of executed probes."""
        if not observed_dict:
            return float(self.mu_joint[4]), float(np.sqrt(self.sigma_joint[4, 4]))

        obs_indices = sorted(list(observed_dict.keys()))
        obs_vals = np.array([observed_dict[i] for i in obs_indices])

        mu_obs = self.mu_joint[obs_indices]
        mu_target = self.mu_joint[4]

        sigma_oo = self.sigma_joint[np.ix_(obs_indices, obs_indices)]
        sigma_to = self.sigma_joint[4, obs_indices]
        sigma_tt = self.sigma_joint[4, 4]

        sigma_oo_inv = np.linalg.pinv(sigma_oo)
        cond_mean = mu_target + sigma_to @ sigma_oo_inv @ (obs_vals - mu_obs)
        cond_var = max(1e-4, sigma_tt - sigma_to @ sigma_oo_inv @ sigma_to.T)

        return float(cond_mean), float(np.sqrt(cond_var))

    def select_next_probe(self, executed_probes: List[int], max_budget: int = 2) -> int:
        """Greedy Expected Information Gain (Max Variance Reduction on Target)."""
        if len(executed_probes) >= max_budget:
            return -1  # Stop: budget exhausted

        candidate_probes = [i for i in range(4) if i not in executed_probes]
        best_probe = -1
        min_expected_var = float("inf")

        for cand in candidate_probes:
            trial_indices = sorted(executed_probes + [cand])
            sigma_oo = self.sigma_joint[np.ix_(trial_indices, trial_indices)]
            sigma_to = self.sigma_joint[4, trial_indices]
            sigma_tt = self.sigma_joint[4, 4]

            # Conditional variance does not depend on observation value (property of Gaussian)
            cond_var = sigma_tt - sigma_to @ np.linalg.pinv(sigma_oo) @ sigma_to.T
            if cond_var < min_expected_var:
                min_expected_var = cond_var
                best_probe = cand

        return best_probe


# ==============================================================================
# 3. TASK CORPUS (TRAIN + STRICT CROSS-TEMPLATE OOD)
# ==============================================================================

def get_datasets():
    train_tasks = [
        {"id": "TR01", "clean": "The Eiffel Tower is located in the city of", "correct": " Paris", "incorrect": " Rome"},
        {"id": "TR02", "clean": "The capital of Japan is the bustling city of", "correct": " Tokyo", "incorrect": " Cairo"},
        {"id": "TR03", "clean": "The famous Big Ben clock tower is located in", "correct": " London", "incorrect": " York"},
        {"id": "TR04", "clean": "The capital city of Germany is known as", "correct": " Berlin", "incorrect": " Paris"},
        {"id": "TR05", "clean": "Alice handed a red apple to Bob. The apple was received by", "correct": " Bob", "incorrect": " Alice"},
        {"id": "TR06", "clean": "John sent a private email to Mary. The email reached", "correct": " Mary", "incorrect": " John"},
        {"id": "TR07", "clean": "David passed the heavy book to Sarah. The book was taken by", "correct": " Sarah", "incorrect": " David"},
        {"id": "TR08", "clean": "While Madrid is famous for soccer, the capital of France is the city of", "correct": " Paris", "incorrect": " Madrid"},
        {"id": "TR09", "clean": "Although Tokyo is very large, the ancient Parthenon is located in", "correct": " Athens", "incorrect": " Tokyo"},
        {"id": "TR10", "clean": "The official capital of Germany is definitely not Rome, it is", "correct": " Berlin", "incorrect": " Rome"}
    ]

    test_tasks = [
        {"id": "TE01_Louvre", "clean": "Which city holds the Louvre Museum? The answer is clearly", "correct": " Paris", "incorrect": " Madrid"},
        {"id": "TE02_Spain", "clean": "If you travel to Spain, you will find its magnificent capital", "correct": " Madrid", "incorrect": " Rome"},
        {"id": "TE03_Binding", "clean": "Tom tossed the baseball to Jack. Who ended up holding the baseball? It was", "correct": " Jack", "incorrect": " Tom"},
        {"id": "TE04_Astronomy", "clean": "In solar astronomy, orbiting closest to the sun is the hot planet", "correct": " Mercury", "incorrect": " Jupiter"},
        {"id": "TE05_Pyramids", "clean": "Despite the fame of Rome, the ancient Pyramids are preserved in", "correct": " Egypt", "incorrect": " Cairo"},
        {"id": "TE06_Distractor", "clean": "Berlin has monuments, yet the ancient Colosseum stands in", "correct": " Rome", "incorrect": " Berlin"}
    ]

    return train_tasks, test_tasks


# ==============================================================================
# 4. MASTER EXPERIMENTAL RUNNER
# ==============================================================================

def run_milestone_7_10():
    harness = M7_10_Harness()
    train_tasks, test_tasks = get_datasets()

    for item in train_tasks + test_tasks:
        _ = harness.model.to_single_token(item["correct"])
        _ = harness.model.to_single_token(item["incorrect"])
    print(f"✅ Verified token integrity across {len(train_tasks)} train and {len(test_tasks)} cross-template test tasks.")

    # 1. Collect full training matrix for Bayesian Joint Prior
    print("\n📍 GATHERING MECHANISTIC PROBE PRIORS ON TRAINING SET...")
    train_records = []
    for t in train_tasks:
        nom_d, target_d, z_orig = harness.get_clean_and_target(t)
        p_resps = [harness.execute_single_probe(t, p_id, nom_d, z_orig) for p_id in range(4)]
        train_records.append({"task_id": t["id"], "probes": p_resps, "target": target_d})

    self_model = BayesianProbeSelfModel(train_records)

    # 2. Benchmark on 6 Strict OOD Tasks
    print("\n" + "=" * 130)
    print("🔬 MILESTONE 7.10: ADAPTIVE PROBING vs FIXED POLICIES (OOD EVALUATION)")
    print("=" * 130)

    policies = ["Zero_Probe_Baseline", "Random_Probe_K1", "Fixed_P2_K1", "Fixed_Optimal_P2P1_K2", "Adaptive_Policy_K2", "Full_Suite_Oracle_K4"]
    policy_results = {p: {"errors": [], "sign_matches": [], "probes_used": []} for p in policies}

    y_actuals = []
    policy_predictions = {p: [] for p in policies}

    probe_names = ["P1(ScaleDown)", "P2(ScaleUp)", "P3(Ortho)", "P4(Damping)"]

    for t in test_tasks:
        nom_d, target_d, z_orig = harness.get_clean_and_target(t)
        y_actuals.append(target_d)

        # A. Zero-Probe
        pred_z, _ = self_model.predict_given_observed({})
        policy_predictions["Zero_Probe_Baseline"].append(pred_z)
        policy_results["Zero_Probe_Baseline"]["errors"].append(abs(target_d - pred_z))
        policy_results["Zero_Probe_Baseline"]["sign_matches"].append(1 if (target_d * pred_z > 0 or abs(target_d) < 0.15) else 0)
        policy_results["Zero_Probe_Baseline"]["probes_used"].append(0)

        # B. Fixed P2 (K=1)
        r_p2 = harness.execute_single_probe(t, 1, nom_d, z_orig)
        pred_p2, _ = self_model.predict_given_observed({1: r_p2})
        policy_predictions["Fixed_P2_K1"].append(pred_p2)
        policy_results["Fixed_P2_K1"]["errors"].append(abs(target_d - pred_p2))
        policy_results["Fixed_P2_K1"]["sign_matches"].append(1 if (target_d * pred_p2 > 0 or abs(target_d) < 0.15) else 0)
        policy_results["Fixed_P2_K1"]["probes_used"].append(1)

        # C. Fixed Optimal P2 -> P1 (K=2)
        r_p1 = harness.execute_single_probe(t, 0, nom_d, z_orig)
        pred_p2p1, _ = self_model.predict_given_observed({1: r_p2, 0: r_p1})
        policy_predictions["Fixed_Optimal_P2P1_K2"].append(pred_p2p1)
        policy_results["Fixed_Optimal_P2P1_K2"]["errors"].append(abs(target_d - pred_p2p1))
        policy_results["Fixed_Optimal_P2P1_K2"]["sign_matches"].append(1 if (target_d * pred_p2p1 > 0 or abs(target_d) < 0.15) else 0)
        policy_results["Fixed_Optimal_P2P1_K2"]["probes_used"].append(2)

        # D. Random Probe (K=1, seeded)
        rand_p_id = (abs(hash(t["id"])) % 4)
        r_rand = harness.execute_single_probe(t, rand_p_id, nom_d, z_orig) if rand_p_id not in [0, 1] else (r_p1 if rand_p_id == 0 else r_p2)
        pred_rand, _ = self_model.predict_given_observed({rand_p_id: r_rand})
        policy_predictions["Random_Probe_K1"].append(pred_rand)
        policy_results["Random_Probe_K1"]["errors"].append(abs(target_d - pred_rand))
        policy_results["Random_Probe_K1"]["sign_matches"].append(1 if (target_d * pred_rand > 0 or abs(target_d) < 0.15) else 0)
        policy_results["Random_Probe_K1"]["probes_used"].append(1)

        # E. Adaptive Sequential Policy (K <= 2)
        executed_adaptive = []
        observed_adaptive = {}
        for step in range(2):
            next_p = self_model.select_next_probe(executed_adaptive, max_budget=2)
            if next_p == -1:
                break
            # Fetch or execute probe
            if next_p == 0: val = r_p1
            elif next_p == 1: val = r_p2
            else: val = harness.execute_single_probe(t, next_p, nom_d, z_orig)

            executed_adaptive.append(next_p)
            observed_adaptive[next_p] = val

            # Early Exit Check: if uncertainty drops below 0.15 variance
            _, curr_std = self_model.predict_given_observed(observed_adaptive)
            if curr_std < 0.35:
                break

        pred_adapt, _ = self_model.predict_given_observed(observed_adaptive)
        policy_predictions["Adaptive_Policy_K2"].append(pred_adapt)
        policy_results["Adaptive_Policy_K2"]["errors"].append(abs(target_d - pred_adapt))
        policy_results["Adaptive_Policy_K2"]["sign_matches"].append(1 if (target_d * pred_adapt > 0 or abs(target_d) < 0.15) else 0)
        policy_results["Adaptive_Policy_K2"]["probes_used"].append(len(executed_adaptive))

        # F. Full Suite Oracle (K=4)
        r_p3 = harness.execute_single_probe(t, 2, nom_d, z_orig)
        r_p4 = harness.execute_single_probe(t, 3, nom_d, z_orig)
        pred_full, _ = self_model.predict_given_observed({0: r_p1, 1: r_p2, 2: r_p3, 3: r_p4})
        policy_predictions["Full_Suite_Oracle_K4"].append(pred_full)
        policy_results["Full_Suite_Oracle_K4"]["errors"].append(abs(target_d - pred_full))
        policy_results["Full_Suite_Oracle_K4"]["sign_matches"].append(1 if (target_d * pred_full > 0 or abs(target_d) < 0.15) else 0)
        policy_results["Full_Suite_Oracle_K4"]["probes_used"].append(4)

        adapt_seq_str = " -> ".join([probe_names[i] for i in executed_adaptive])
        print(f"Task: {t['id']:<15} | Actual: {target_d:+6.3f} | Adapt Pred: {pred_adapt:+6.3f} | Probes: [{adapt_seq_str}]")

    # -------------------------------------------------------------------------
    # 5. SUMMARY COMPARISON TABLE
    # -------------------------------------------------------------------------
    print("\n" + "=" * 130)
    print("📊 POLICY PERFORMANCE & EFFICIENCY AUDIT (MILESTONE 7.10)")
    print("=" * 130)
    print(f"{'Exploration Strategy':<30} | {'Avg Probes (K)':<16} | {'OOD MAE (Logits)':<18} | {'Sign Acc (%)':<16} | {'Spearman ρ'}")
    print("-" * 130)

    summary_telemetry = {}

    for p in policies:
        avg_k = float(np.mean(policy_results[p]["probes_used"]))
        mae = float(np.mean(policy_results[p]["errors"]))
        sign_acc = float(np.mean(policy_results[p]["sign_matches"]) * 100.0)
        rho, _ = spearmanr(policy_predictions[p], y_actuals)

        print(f"{p:<30} | {avg_k:12.2f}    | {mae:14.4f}     | {sign_acc:12.2f}%    | {rho:+10.4f}")

        summary_telemetry[p] = {
            "avg_probes_used": avg_k,
            "mae": mae,
            "sign_accuracy": sign_acc,
            "spearman_rho": float(rho)
        }

    print("=" * 130)
    adapt_mae = summary_telemetry["Adaptive_Policy_K2"]["mae"]
    oracle_mae = summary_telemetry["Full_Suite_Oracle_K4"]["mae"]
    rand_mae = summary_telemetry["Random_Probe_K1"]["mae"]

    if adapt_mae <= oracle_mae * 1.08 and adapt_mae < rand_mae:
        print("👉 VERDICT: 🟢 ADAPTIVE_SELF_PROBING_VALIDATED (Information Gain policy achieves Oracle parity at fraction of cost)")
    else:
        print("👉 VERDICT: 🟡 FIXED_OR_RANDOM_COMPETITIVE")
    print("=" * 130 + "\n")

    with open("m7_10_adaptive_probing_result.json", "w", encoding="utf-8") as f:
        json.dump({
            "milestone": "7.10_ADAPTIVE_SELF_PROBING",
            "telemetry": summary_telemetry
        }, f, indent=2, ensure_ascii=False)
    print("📁 Comprehensive telemetry saved to: m7_10_adaptive_probing_result.json")


if __name__ == "__main__":
    run_milestone_7_10()

🔄 Loading Qwen/Qwen2.5-0.5B on CUDA (bfloat16)...


/tmp/ipykernel_2736/4164165410.py:38: DeprecationWarning: HookedTransformer.from_pretrained is deprecated and will be removed in a future major release. Use TransformerBridge.boot_transformers(...) instead, then call enable_compatibility_mode() for HookedTransformer-equivalent numerics. See docs/source/content/migrating_to_v3.md.
  self.model = HookedTransformer.from_pretrained(


config.json:   0%|          | 0.00/681 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  988MB            

model.safetensors: downloading bytes:           |  0.00B            

...

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

Loaded pretrained model Qwen/Qwen2.5-0.5B into HookedTransformer
✅ Verified token integrity across 10 train and 6 cross-template test tasks.

📍 GATHERING MECHANISTIC PROBE PRIORS ON TRAINING SET...

🔬 MILESTONE 7.10: ADAPTIVE PROBING vs FIXED POLICIES (OOD EVALUATION)
Task: TE01_Louvre     | Actual: +0.375 | Adapt Pred: +0.036 | Probes: [P2(ScaleUp)]
Task: TE02_Spain      | Actual: +1.688 | Adapt Pred: +2.500 | Probes: [P2(ScaleUp)]
Task: TE03_Binding    | Actual: +0.125 | Adapt Pred: +0.036 | Probes: [P2(ScaleUp)]
Task: TE04_Astronomy  | Actual: +0.250 | Adapt Pred: +0.036 | Probes: [P2(ScaleUp)]
Task: TE05_Pyramids   | Actual: -0.562 | Adapt Pred: -0.786 | Probes: [P2(ScaleUp)]
Task: TE06_Distractor | Actual: -1.750 | Adapt Pred: -0.786 | Probes: [P2(ScaleUp)]

📊 POLICY PERFORMANCE & EFFICIENCY AUDIT (MILESTONE 7.10)
Exploration Strategy           | Avg Probes (K)   | OOD MAE (Logits)   | Sign Acc (%)     | Spearman ρ
------------------------------------------------------------------

/tmp/ipykernel_2736/4164165410.py:332: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  rho, _ = spearmanr(policy_predictions[p], y_actuals)


In [3]:
"""
m8_1_formal_epistemic_state.py
===============================
Milestone 8.1: Formal Epistemic Self-Model State & Multi-Component Uncertainty Engine.

Components Tracked:
  - L21H06, L21H01, L17H01, L14H04, L10H00
"""

import torch
import numpy as np
import json
from dataclasses import dataclass, field
from typing import Dict, List, Tuple, Optional
from transformer_lens import HookedTransformer

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)


@dataclass
class EpistemicBelief:
    component: str
    expected_damage: float = 0.0
    aleatoric_var: float = 0.50
    evidence_vectors: List[List[float]] = field(default_factory=list)
    probe_predictions: Dict[str, float] = field(default_factory=dict)

    def compute_uncertainty(self, current_context: torch.Tensor) -> Dict[str, float]:
        ctx_np = current_context.detach().cpu().to(torch.float32).numpy()

        # 1. Epistemic Uncertainty (Evidence density)
        if not self.evidence_vectors:
            u_epistemic = 1.0
            u_novelty = 1.0
        else:
            sims = [
                float(np.dot(ctx_np, ev) / (np.linalg.norm(ctx_np) * np.linalg.norm(ev) + 1e-8))
                for ev in self.evidence_vectors
            ]
            effective_evidence = sum(max(0.0, s) for s in sims)
            u_epistemic = float(np.exp(-0.40 * effective_evidence))
            u_novelty = float(max(0.0, 1.0 - max(sims)))

        # 2. Probe Disagreement
        if len(self.probe_predictions) >= 2:
            vals = list(self.probe_predictions.values())
            u_disagree = float(np.var(vals))
        else:
            u_disagree = 0.50

        u_total = float(0.40 * u_epistemic + 0.35 * u_novelty + 0.25 * min(1.0, u_disagree))

        return {
            "u_total": u_total,
            "u_epistemic": u_epistemic,
            "u_novelty": u_novelty,
            "u_disagree": u_disagree
        }


class MultiComponentSelfModel:
    def __init__(self, tracked_components: List[Tuple[int, int]]):
        self.components = tracked_components
        self.beliefs: Dict[str, EpistemicBelief] = {
            f"L{l:02d}H{h:02d}": EpistemicBelief(component=f"L{l:02d}H{h:02d}")
            for (l, h) in tracked_components
        }

    def assess_state(self, current_contexts: Dict[str, torch.Tensor]) -> Dict[str, Dict]:
        report = {}
        for comp_name, belief in self.beliefs.items():
            ctx = current_contexts[comp_name]
            unc = belief.compute_uncertainty(ctx)
            report[comp_name] = {
                "expected_damage": belief.expected_damage,
                "uncertainty": unc,
                "evidence_count": len(belief.evidence_vectors)
            }
        return report

    def update_evidence(self, comp_name: str, damage: float, context: torch.Tensor, probe_preds: Dict[str, float]):
        belief = self.beliefs[comp_name]
        ctx_np = context.detach().cpu().to(torch.float32).numpy().tolist()
        belief.evidence_vectors.append(ctx_np)
        belief.expected_damage = 0.70 * belief.expected_damage + 0.30 * damage
        belief.probe_predictions = probe_preds


class M8_1_Harness:
    def __init__(self, model_name: str = "Qwen/Qwen2.5-0.5B", device: str = "cuda"):
        self.device = device if torch.cuda.is_available() else "cpu"
        self.model = HookedTransformer.from_pretrained(
            model_name, device=self.device, torch_dtype=torch.bfloat16, default_padding_side="left"
        )
        self.model.eval()
        self.tracked_components = [(21, 6), (21, 1), (17, 1), (14, 4), (10, 0)]

    @torch.no_grad()
    def extract_component_states(self, prompt: str) -> Dict[str, torch.Tensor]:
        tokens = self.model.to_tokens(prompt)
        _, cache = self.model.run_with_cache(tokens)
        states = {}
        for (l, h) in self.tracked_components:
            z = cache[f"blocks.{l}.attn.hook_z"][0, -1, h, :].clone()
            states[f"L{l:02d}H{h:02d}"] = z
        return states


def run_milestone_8_1_demo():
    harness = M8_1_Harness()
    self_model = MultiComponentSelfModel(harness.tracked_components)

    test_prompt_clean = "The capital of France is the city of"
    test_prompt_ood = "Which ancient civilization constructed the Great Sphinx in"

    print("===================================================================================================")
    print("🔬 MILESTONE 8.1: FORMAL EPISTEMIC SELF-MODEL STATE & MULTI-ORGAN UNCERTAINTY")
    print("===================================================================================================")

    # 1. Initial State (Tabula Rasa)
    states_clean = harness.extract_component_states(test_prompt_clean)
    report_initial = self_model.assess_state(states_clean)

    print(f"\n📍 Initial Assessment (Zero Evidence): Prompt = '{test_prompt_clean}'")
    print(f"{'Component':<12} | {'Expected Damage':<16} | {'U_Total':<10} | {'U_Epistemic':<12} | {'U_Novelty':<10}")
    print("-" * 75)
    for c_name, data in report_initial.items():
        unc = data["uncertainty"]
        print(f"{c_name:<12} | {data['expected_damage']:+14.3f}   | {unc['u_total']:8.3f}   | {unc['u_epistemic']:10.3f}   | {unc['u_novelty']:8.3f}")

    # 2. Inject Evidence for L21H06 and L17H01
    self_model.update_evidence("L21H06", damage=1.85, context=states_clean["L21H06"], probe_preds={"P1": 1.70, "P2": 1.95})
    self_model.update_evidence("L17H01", damage=0.90, context=states_clean["L17H01"], probe_preds={"P1": 0.85, "P2": 0.95})

    # 3. Assessment on Familiar Context
    report_after = self_model.assess_state(states_clean)
    print(f"\n📍 Assessment After Clean Evidence: Prompt = '{test_prompt_clean}'")
    print(f"{'Component':<12} | {'Expected Damage':<16} | {'U_Total':<10} | {'U_Epistemic':<12} | {'U_Novelty':<10}")
    print("-" * 75)
    for c_name, data in report_after.items():
        unc = data["uncertainty"]
        print(f"{c_name:<12} | {data['expected_damage']:+14.3f}   | {unc['u_total']:8.3f}   | {unc['u_epistemic']:10.3f}   | {unc['u_novelty']:8.3f}")

    # 4. Assessment on OOD Context
    states_ood = harness.extract_component_states(test_prompt_ood)
    report_ood = self_model.assess_state(states_ood)
    print(f"\n📍 Assessment on OOD Context: Prompt = '{test_prompt_ood}'")
    print(f"{'Component':<12} | {'Expected Damage':<16} | {'U_Total':<10} | {'U_Epistemic':<12} | {'U_Novelty':<10}")
    print("-" * 75)
    for c_name, data in report_ood.items():
        unc = data["uncertainty"]
        print(f"{c_name:<12} | {data['expected_damage']:+14.3f}   | {unc['u_total']:8.3f}   | {unc['u_epistemic']:10.3f}   | {unc['u_novelty']:8.3f}")

    print("\n" + "=" * 100)


if __name__ == "__main__":
    run_milestone_8_1_demo()

/tmp/ipykernel_2736/1376596658.py:94: DeprecationWarning: HookedTransformer.from_pretrained is deprecated and will be removed in a future major release. Use TransformerBridge.boot_transformers(...) instead, then call enable_compatibility_mode() for HookedTransformer-equivalent numerics. See docs/source/content/migrating_to_v3.md.
  self.model = HookedTransformer.from_pretrained(


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Loaded pretrained model Qwen/Qwen2.5-0.5B into HookedTransformer
🔬 MILESTONE 8.1: FORMAL EPISTEMIC SELF-MODEL STATE & MULTI-ORGAN UNCERTAINTY

📍 Initial Assessment (Zero Evidence): Prompt = 'The capital of France is the city of'
Component    | Expected Damage  | U_Total    | U_Epistemic  | U_Novelty 
---------------------------------------------------------------------------
L21H06       |         +0.000   |    0.875   |      1.000   |    1.000
L21H01       |         +0.000   |    0.875   |      1.000   |    1.000
L17H01       |         +0.000   |    0.875   |      1.000   |    1.000
L14H04       |         +0.000   |    0.875   |      1.000   |    1.000
L10H00       |         +0.000   |    0.875   |      1.000   |    1.000

📍 Assessment After Clean Evidence: Prompt = 'The capital of France is the city of'
Component    | Expected Damage  | U_Total    | U_Epistemic  | U_Novelty 
---------------------------------------------------------------------------
L21H06       |         +0.555   | 

In [4]:
"""
m8_2_active_information_gain_engine.py
======================================
Milestone 8.2: Multi-Organ Active Epistemic Inquiry via Expected Information Gain (EIG).

Simultaneously selects:
  1. WHICH Component to probe (L21H06, L21H01, L17H01, L14H04, L10H00)
  2. WHICH Probe to execute (P1: ScaleDown, P2: ScaleUp, P3: Orthogonal, P4: Damping)

Based on Expected Uncertainty Reduction (EIG), not fixed heuristics.
"""

import torch
import numpy as np
import json
from dataclasses import dataclass, field
from typing import Dict, List, Tuple, Optional
from transformer_lens import HookedTransformer

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)


# ==============================================================================
# 1. FORMAL EPISTEMIC MULTI-ORGAN SELF-MODEL
# ==============================================================================

@dataclass
class EpistemicOrganBelief:
    component_name: str
    layer: int
    head: int
    expected_damage: float = 0.0
    aleatoric_var: float = 0.50
    evidence_contexts: List[List[float]] = field(default_factory=list)
    probe_response_history: Dict[str, List[float]] = field(default_factory=lambda: {"P1": [], "P2": [], "P3": [], "P4": []})

    def get_uncertainty(self, ctx_tensor: torch.Tensor) -> Dict[str, float]:
        ctx_np = ctx_tensor.detach().cpu().to(torch.float32).numpy()

        # 1. Epistemic evidence density
        if not self.evidence_contexts:
            u_epistemic = 1.0
            u_novelty = 1.0
        else:
            sims = [
                float(np.dot(ctx_np, ev) / (np.linalg.norm(ctx_np) * np.linalg.norm(ev) + 1e-8))
                for ev in self.evidence_contexts
            ]
            effective_evidence = sum(max(0.0, s) for s in sims)
            u_epistemic = float(np.exp(-0.45 * effective_evidence))
            u_novelty = float(max(0.0, 1.0 - max(sims)))

        # 2. Probe Disagreement & Empirical Variance
        all_resps = []
        for p_list in self.probe_response_history.values():
            all_resps.extend(p_list)

        if len(all_resps) >= 3:
            u_disagree = float(min(1.0, np.var(all_resps)))
        else:
            u_disagree = 0.60

        u_total = float(0.40 * u_epistemic + 0.35 * u_novelty + 0.25 * u_disagree)
        return {"u_total": u_total, "u_epistemic": u_epistemic, "u_novelty": u_novelty, "u_disagree": u_disagree}

    def compute_probe_eig(self, probe_name: str, ctx_tensor: torch.Tensor) -> float:
        """Estimates Expected Information Gain for executing probe on this specific component."""
        unc = self.get_uncertainty(ctx_tensor)
        n_obs = len(self.probe_response_history[probe_name])

        # Diminishing returns per probe type on the same component
        probe_novelty_factor = np.exp(-0.50 * n_obs)

        # Expected variance reduction is proportional to current uncertainty * probe novelty
        expected_reduction = unc["u_total"] * (0.60 * unc["u_novelty"] + 0.40 * probe_novelty_factor)
        return float(expected_reduction)


class EpistemicInquiryPlanner:
    def __init__(self, tracked_heads: List[Tuple[int, int]]):
        self.organs: Dict[str, EpistemicOrganBelief] = {
            f"L{l:02d}H{h:02d}": EpistemicOrganBelief(component_name=f"L{l:02d}H{h:02d}", layer=l, head=h)
            for (l, h) in tracked_heads
        }
        self.probe_cost = {"P1": 0.05, "P2": 0.05, "P3": 0.08, "P4": 0.05}

    def select_optimal_inquiry(self, current_states: Dict[str, torch.Tensor]) -> Tuple[str, str, float]:
        """Greedy selection of (Organ, Probe) maximizing Expected Information Gain - Cost."""
        best_organ, best_probe, max_net_eig = None, None, -float("inf")

        for organ_name, organ in self.organs.items():
            ctx = current_states[organ_name]
            for p_name in ["P1", "P2", "P3", "P4"]:
                eig = organ.compute_probe_eig(p_name, ctx)
                net_value = eig - self.probe_cost[p_name]
                if net_value > max_net_eig:
                    max_net_eig = net_value
                    best_organ = organ_name
                    best_probe = p_name

        return best_organ, best_probe, max_net_eig

    def update_with_result(self, organ_name: str, probe_name: str, response: float, ctx_tensor: torch.Tensor):
        organ = self.organs[organ_name]
        ctx_np = ctx_tensor.detach().cpu().to(torch.float32).numpy().tolist()
        organ.evidence_contexts.append(ctx_np)
        organ.probe_response_history[probe_name].append(response)


# ==============================================================================
# 2. MECHANISTIC HARNESS
# ==============================================================================

class M8_2_Harness:
    def __init__(self, model_name: str = "Qwen/Qwen2.5-0.5B", device: str = "cuda"):
        self.device = device if torch.cuda.is_available() else "cpu"
        print(f"🔄 Loading {model_name} on {self.device.upper()} (bfloat16)...")
        self.model = HookedTransformer.from_pretrained(
            model_name, device=self.device, torch_dtype=torch.bfloat16, default_padding_side="left"
        )
        self.model.eval()
        self.tracked_heads = [(21, 6), (21, 1), (17, 1), (14, 4), (10, 0)]
        self.d_head = self.model.cfg.d_head
        v_rand = torch.randn(self.d_head, dtype=torch.float32)
        self.v_ortho = (v_rand / v_rand.norm()).to(self.device)

    def compute_diff(self, logits: torch.Tensor, c_id: int, inc_id: int) -> float:
        return float((logits[0, -1, c_id] - logits[0, -1, inc_id]).detach().cpu().item())

    @torch.no_grad()
    def get_states_and_nom(self, task: Dict) -> Tuple[float, Dict[str, torch.Tensor]]:
        clean_toks = self.model.to_tokens(task["clean"])
        c_id = int(self.model.to_single_token(task["correct"]))
        inc_id = int(self.model.to_single_token(task["incorrect"]))

        nom_logits, cache = self.model.run_with_cache(clean_toks)
        nom_diff = self.compute_diff(nom_logits, c_id, inc_id)

        states = {}
        for (l, h) in self.tracked_heads:
            states[f"L{l:02d}H{h:02d}"] = cache[f"blocks.{l}.attn.hook_z"][0, -1, h, :].clone()
        return nom_diff, states

    @torch.no_grad()
    def execute_probe(self, task: Dict, layer: int, head: int, probe_name: str, nom_diff: float) -> float:
        clean_toks = self.model.to_tokens(task["clean"])
        c_id = int(self.model.to_single_token(task["correct"]))
        inc_id = int(self.model.to_single_token(task["incorrect"]))

        if probe_name == "P1":  # Scale Down
            def hook_fn(act, hook): act[:, -1, head, :] *= 0.90; return act
        elif probe_name == "P2":  # Scale Up
            def hook_fn(act, hook): act[:, -1, head, :] *= 1.10; return act
        elif probe_name == "P3":  # Orthogonal Jitter
            def hook_fn(act, hook): act[:, -1, head, :] += 0.06 * act[:, -1, head, :].norm() * self.v_ortho.to(act.dtype); return act
        elif probe_name == "P4":  # Damping
            def hook_fn(act, hook): act[:, -1, head, :] -= 0.08 * act[:, -1, head, :]; return act
        else:
            raise ValueError(f"Unknown probe: {probe_name}")

        with self.model.hooks(fwd_hooks=[(f"blocks.{layer}.attn.hook_z", hook_fn)]):
            p_logits, _ = self.model.run_with_cache(clean_toks)
        return float(nom_diff - self.compute_diff(p_logits, c_id, inc_id))


# ==============================================================================
# 3. SEQUENTIAL ACTIVE DISCOVERY TRACE
# ==============================================================================

def run_milestone_8_2():
    harness = M8_2_Harness()
    planner = EpistemicInquiryPlanner(harness.tracked_heads)

    task_corpus = [
        {"id": "T1_FactParis", "clean": "The Eiffel Tower is located in the city of", "correct": " Paris", "incorrect": " Rome"},
        {"id": "T2_BindingBob", "clean": "Alice handed a red apple to Bob. The apple was received by", "correct": " Bob", "incorrect": " Alice"},
        {"id": "T3_DistractorRome", "clean": "Berlin has monuments, yet the ancient Colosseum stands in", "correct": " Rome", "incorrect": " Berlin"},
        {"id": "T4_FactTokyo", "clean": "The capital of Japan is the bustling city of", "correct": " Tokyo", "incorrect": " Cairo"},
        {"id": "T5_BindingMary", "clean": "John sent a private email to Mary. The email reached", "correct": " Mary", "incorrect": " John"},
        {"id": "T6_PyramidsOOD", "clean": "Despite the fame of Rome, the ancient Pyramids are preserved in", "correct": " Egypt", "incorrect": " Cairo"}
    ]

    for t in task_corpus:
        _ = harness.model.to_single_token(t["correct"])
        _ = harness.model.to_single_token(t["incorrect"])
    print(f"✅ Verified token integrity across task sequence.")

    print("\n" + "=" * 125)
    print("🔬 MILESTONE 8.2: MULTI-ORGAN SEQUENTIAL ACTIVE INQUIRY TRACE (EIG-DRIVEN)")
    print("=" * 125)
    print(f"{'Step':<5} | {'Task':<16} | {'Target Organ':<14} | {'Selected Probe':<16} | {'Net EIG':<10} | {'Observed Response':<18} | {'Remaining Total Uncertainty'}")
    print("-" * 125)

    trace_logs = []

    for step_idx, task in enumerate(task_corpus, start=1):
        nom_d, states = harness.get_states_and_nom(task)

        # 1. Decide optimal organ and probe via EIG
        target_organ, chosen_probe, net_eig = planner.select_optimal_inquiry(states)
        l, h = planner.organs[target_organ].layer, planner.organs[target_organ].head

        # 2. Execute selected intervention
        resp = harness.execute_probe(task, l, h, chosen_probe, nom_d)

        # 3. Update Self-Model Belief
        planner.update_with_result(target_organ, chosen_probe, resp, states[target_organ])

        # 4. Measure global remaining uncertainty
        curr_uncs = [planner.organs[c].get_uncertainty(states[c])["u_total"] for c in planner.organs]
        mean_global_u = float(np.mean(curr_uncs))

        print(f"#{step_idx:02d}  | {task['id']:<16} | {target_organ:<14} | {chosen_probe:<16} | {net_eig:8.4f}   | {resp:+14.4f}     | {mean_global_u:12.4f}")

        trace_logs.append({
            "step": step_idx,
            "task": task["id"],
            "organ": target_organ,
            "probe": chosen_probe,
            "eig": net_eig,
            "response": resp,
            "global_uncertainty": mean_global_u
        })

    print("=" * 125)

    # Check if policy diversified beyond single probe
    selected_organs = set(r["organ"] for r in trace_logs)
    selected_probes = set(r["probe"] for r in trace_logs)
    print(f"\n📊 EXPLORATION SUMMARY:")
    print(f"  • Unique Organs Explored: {len(selected_organs)} ({list(selected_organs)})")
    print(f"  • Unique Probes Utilized: {len(selected_probes)} ({list(selected_probes)})")

    if len(selected_organs) > 1 and len(selected_probes) > 1:
        print("👉 VERDICT: 🟢 TRUE_ACTIVE_INQUIRY_VALIDATED (Policy dynamically routes across multi-component & multi-probe space)")
    else:
        print("👉 VERDICT: 🔴 DEGENERATE_POLICY")
    print("=" * 125 + "\n")

    with open("m8_2_active_inquiry_trace.json", "w", encoding="utf-8") as f:
        json.dump({"milestone": "8.2_ACTIVE_INQUIRY_EIG", "trace": trace_logs}, f, indent=2, ensure_ascii=False)


if __name__ == "__main__":
    run_milestone_8_2()

🔄 Loading Qwen/Qwen2.5-0.5B on CUDA (bfloat16)...


/tmp/ipykernel_2736/518863193.py:120: DeprecationWarning: HookedTransformer.from_pretrained is deprecated and will be removed in a future major release. Use TransformerBridge.boot_transformers(...) instead, then call enable_compatibility_mode() for HookedTransformer-equivalent numerics. See docs/source/content/migrating_to_v3.md.
  self.model = HookedTransformer.from_pretrained(


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Loaded pretrained model Qwen/Qwen2.5-0.5B into HookedTransformer
✅ Verified token integrity across task sequence.

🔬 MILESTONE 8.2: MULTI-ORGAN SEQUENTIAL ACTIVE INQUIRY TRACE (EIG-DRIVEN)
Step  | Task             | Target Organ   | Selected Probe   | Net EIG    | Observed Response  | Remaining Total Uncertainty
-----------------------------------------------------------------------------------------------------------------------------
#01  | T1_FactParis     | L21H06         | P1               |   0.8500   |        +0.1250     |       0.8010
#02  | T2_BindingBob    | L21H06         | P2               |   0.9235   |        +0.0000     |       0.8010
#03  | T3_DistractorRome | L21H01         | P1               |   0.8500   |        +0.0000     |       0.7678
#04  | T4_FactTokyo     | L17H01         | P1               |   0.8500   |        +0.0000     |       0.7651
#05  | T5_BindingMary   | L14H04         | P1               |   0.8500   |        -0.1250     |       0.6752
#06  | T6_Pyra

In [5]:
"""
m8_2_active_information_gain_v2.py
===================================
Milestone 8.2 (Rewritten): Multi-Organ Active Inquiry with Functional Sensitivity Weighting.

Key Improvements:
  1. Functional Sensitivity: Downweights uninformative/silent control heads after observing near-zero impact.
  2. Extended Horizon (18 Steps): Evaluates Phase 2 (post-initialization exploitation vs exploration).
  3. Falsification Metric: Checks if L21H06 is selectively targeted over silent control heads.
"""

import torch
import numpy as np
import json
from dataclasses import dataclass, field
from typing import Dict, List, Tuple
from transformer_lens import HookedTransformer

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)


# ==============================================================================
# 1. EPISTEMIC BELIEF WITH FUNCTIONAL SENSITIVITY
# ==============================================================================

@dataclass
class EpistemicOrganBeliefV2:
    component_name: str
    layer: int
    head: int
    evidence_contexts: List[List[float]] = field(default_factory=list)
    probe_response_history: Dict[str, List[float]] = field(
        default_factory=lambda: {"P1": [], "P2": [], "P3": [], "P4": []}
    )

    def get_uncertainty(self, ctx_tensor: torch.Tensor) -> Dict[str, float]:
        ctx_np = ctx_tensor.detach().cpu().to(torch.float32).numpy()

        if not self.evidence_contexts:
            u_epistemic = 1.0
            u_novelty = 1.0
        else:
            sims = [
                float(np.dot(ctx_np, ev) / (np.linalg.norm(ctx_np) * np.linalg.norm(ev) + 1e-8))
                for ev in self.evidence_contexts
            ]
            effective_evidence = sum(max(0.0, s) for s in sims)
            u_epistemic = float(np.exp(-0.40 * effective_evidence))
            u_novelty = float(max(0.0, 1.0 - max(sims)))

        # Response variance / disagreement
        all_resps = [r for sub in self.probe_response_history.values() for r in sub]
        if len(all_resps) >= 2:
            u_disagree = float(min(1.0, np.var(all_resps) * 4.0))
        else:
            u_disagree = 0.50

        u_total = float(0.45 * u_epistemic + 0.35 * u_novelty + 0.20 * u_disagree)
        return {"u_total": u_total, "u_epistemic": u_epistemic, "u_novelty": u_novelty, "u_disagree": u_disagree}

    def compute_functional_impact_weight(self) -> float:
        """Estimates causal importance. Dampens priority if head consistently outputs 0.0."""
        all_resps = [abs(r) for sub in self.probe_response_history.values() for r in sub]
        if not all_resps:
            return 1.0  # Optimistic prior for untouched components

        mean_abs_impact = float(np.mean(all_resps))
        max_abs_impact = float(np.max(all_resps))
        # Responsive heads get high weight; inert control heads get scaled down to ~0.15
        impact_score = max(0.15, min(1.5, 0.5 * mean_abs_impact + 0.5 * max_abs_impact * 2.0))
        return impact_score

    def compute_probe_eig(self, probe_name: str, ctx_tensor: torch.Tensor) -> float:
        unc = self.get_uncertainty(ctx_tensor)
        impact_w = self.compute_functional_impact_weight()
        n_obs = len(self.probe_response_history[probe_name])

        # Diminishing returns per probe type on same head
        diminish_factor = float(np.exp(-0.60 * n_obs))

        # Core Formula: Uncertainty * Functional Impact * Diminishing Factor
        raw_eig = unc["u_total"] * impact_w * (0.50 * unc["u_novelty"] + 0.50 * diminish_factor)
        return float(raw_eig)


class AdaptiveInquiryEngineV2:
    def __init__(self, tracked_heads: List[Tuple[int, int]]):
        self.organs: Dict[str, EpistemicOrganBeliefV2] = {
            f"L{l:02d}H{h:02d}": EpistemicOrganBeliefV2(component_name=f"L{l:02d}H{h:02d}", layer=l, head=h)
            for (l, h) in tracked_heads
        }
        self.probe_cost = {"P1": 0.04, "P2": 0.04, "P3": 0.06, "P4": 0.04}

    def select_inquiry(self, current_states: Dict[str, torch.Tensor]) -> Tuple[str, str, float]:
        best_organ, best_probe, max_net = None, None, -float("inf")

        for organ_name, organ in self.organs.items():
            ctx = current_states[organ_name]
            for p_name in ["P1", "P2", "P3", "P4"]:
                eig = organ.compute_probe_eig(p_name, ctx)
                net = eig - self.probe_cost[p_name]
                if net > max_net:
                    max_net = net
                    best_organ = organ_name
                    best_probe = p_name

        return best_organ, best_probe, max_net

    def record_observation(self, organ_name: str, probe_name: str, response: float, ctx_tensor: torch.Tensor):
        organ = self.organs[organ_name]
        organ.evidence_contexts.append(ctx_tensor.detach().cpu().to(torch.float32).numpy().tolist())
        organ.probe_response_history[probe_name].append(response)


# ==============================================================================
# 2. MECHANISTIC HARNESS
# ==============================================================================

class M8_2_HarnessV2:
    def __init__(self, model_name: str = "Qwen/Qwen2.5-0.5B", device: str = "cuda"):
        self.device = device if torch.cuda.is_available() else "cpu"
        print(f"🔄 Loading {model_name} on {self.device.upper()} (bfloat16)...")
        self.model = HookedTransformer.from_pretrained(
            model_name, device=self.device, torch_dtype=torch.bfloat16, default_padding_side="left"
        )
        self.model.eval()
        self.tracked_heads = [(21, 6), (21, 1), (17, 1), (14, 4), (10, 0)]
        self.d_head = self.model.cfg.d_head
        v_rand = torch.randn(self.d_head, dtype=torch.float32)
        self.v_ortho = (v_rand / v_rand.norm()).to(self.device)

    def compute_diff(self, logits: torch.Tensor, c_id: int, inc_id: int) -> float:
        return float((logits[0, -1, c_id] - logits[0, -1, inc_id]).detach().cpu().item())

    @torch.no_grad()
    def get_states_and_nom(self, task: Dict) -> Tuple[float, Dict[str, torch.Tensor]]:
        clean_toks = self.model.to_tokens(task["clean"])
        c_id = int(self.model.to_single_token(task["correct"]))
        inc_id = int(self.model.to_single_token(task["incorrect"]))

        nom_logits, cache = self.model.run_with_cache(clean_toks)
        nom_diff = self.compute_diff(nom_logits, c_id, inc_id)

        states = {
            f"L{l:02d}H{h:02d}": cache[f"blocks.{l}.attn.hook_z"][0, -1, h, :].clone()
            for (l, h) in self.tracked_heads
        }
        return nom_diff, states

    @torch.no_grad()
    def execute_probe(self, task: Dict, layer: int, head: int, probe_name: str, nom_diff: float) -> float:
        clean_toks = self.model.to_tokens(task["clean"])
        c_id = int(self.model.to_single_token(task["correct"]))
        inc_id = int(self.model.to_single_token(task["incorrect"]))

        if probe_name == "P1":
            def hook_fn(act, hook): act[:, -1, head, :] *= 0.90; return act
        elif probe_name == "P2":
            def hook_fn(act, hook): act[:, -1, head, :] *= 1.10; return act
        elif probe_name == "P3":
            def hook_fn(act, hook): act[:, -1, head, :] += 0.06 * act[:, -1, head, :].norm() * self.v_ortho.to(act.dtype); return act
        elif probe_name == "P4":
            def hook_fn(act, hook): act[:, -1, head, :] -= 0.08 * act[:, -1, head, :]; return act
        else:
            raise ValueError(f"Unknown probe: {probe_name}")

        with self.model.hooks(fwd_hooks=[(f"blocks.{layer}.attn.hook_z", hook_fn)]):
            p_logits, _ = self.model.run_with_cache(clean_toks)
        return float(nom_diff - self.compute_diff(p_logits, c_id, inc_id))


# ==============================================================================
# 3. EXTENDED 18-STEP EXECUTION BENCHMARK
# ==============================================================================

def get_extended_corpus():
    return [
        # Phase 1: Initial exploration mix
        {"id": "T01_FactParis", "clean": "The Eiffel Tower is located in the city of", "correct": " Paris", "incorrect": " Rome"},
        {"id": "T02_BindingBob", "clean": "Alice handed a red apple to Bob. The apple was received by", "correct": " Bob", "incorrect": " Alice"},
        {"id": "T03_DistractorRome", "clean": "Berlin has monuments, yet the ancient Colosseum stands in", "correct": " Rome", "incorrect": " Berlin"},
        {"id": "T04_FactTokyo", "clean": "The capital of Japan is the bustling city of", "correct": " Tokyo", "incorrect": " Cairo"},
        {"id": "T05_BindingMary", "clean": "John sent a private email to Mary. The email reached", "correct": " Mary", "incorrect": " John"},
        {"id": "T06_PyramidsEgypt", "clean": "Despite the fame of Rome, the ancient Pyramids are preserved in", "correct": " Egypt", "incorrect": " Cairo"},

        # Phase 2: Extended tests to evaluate component specialization
        {"id": "T07_FactLondon", "clean": "The famous Big Ben clock tower is located in", "correct": " London", "incorrect": " York"},
        {"id": "T08_BindingSarah", "clean": "David passed the heavy book to Sarah. The book was taken by", "correct": " Sarah", "incorrect": " David"},
        {"id": "T09_DistractorParis", "clean": "While Madrid is famous for soccer, the capital of France is", "correct": " Paris", "incorrect": " Madrid"},
        {"id": "T10_FactBerlin", "clean": "The official capital city of Germany is", "correct": " Berlin", "incorrect": " Munich"},
        {"id": "T11_BindingEmma", "clean": "Michael offered a cup of tea to Emma. The tea was accepted by", "correct": " Emma", "incorrect": " Michael"},
        {"id": "T12_DistractorAthens", "clean": "Although Tokyo is very large, the ancient Parthenon is located in", "correct": " Athens", "incorrect": " Tokyo"},
        {"id": "T13_FactMoscow", "clean": "The historic Kremlin is located in the city of", "correct": " Moscow", "incorrect": " Athens"},
        {"id": "T14_BindingJack", "clean": "Tom tossed the baseball to Jack. The ball was caught by", "correct": " Jack", "incorrect": " Tom"},
        {"id": "T15_DistractorRome2", "clean": "Even though Cairo has pyramids, the capital of Italy is", "correct": " Rome", "incorrect": " Cairo"},
        {"id": "T16_FactMadrid", "clean": "The city serving as the capital of Spain is definitely", "correct": " Madrid", "incorrect": " Rome"},
        {"id": "T17_BindingLisa", "clean": "Robert wrote a letter to Lisa. The letter was delivered to", "correct": " Lisa", "incorrect": " Robert"},
        {"id": "T18_DistractorBerlin2", "clean": "The official capital of Germany is definitely not Rome, it is", "correct": " Berlin", "incorrect": " Rome"}
    ]


def run_milestone_8_2_v2():
    harness = M8_2_HarnessV2()
    planner = AdaptiveInquiryEngineV2(harness.tracked_heads)
    corpus = get_extended_corpus()

    for t in corpus:
        _ = harness.model.to_single_token(t["correct"])
        _ = harness.model.to_single_token(t["incorrect"])
    print(f"✅ Verified token integrity across {len(corpus)} extended tasks.")

    print("\n" + "=" * 135)
    print("🔬 MILESTONE 8.2 (V2): EXTENDED 18-STEP ACTIVE INQUIRY TRACE WITH FUNCTIONAL WEIGHTING")
    print("=" * 135)
    print(f"{'Step':<5} | {'Task ID':<20} | {'Organ':<10} | {'Probe':<8} | {'Net EIG':<10} | {'Impact W':<10} | {'Observed Resp':<16} | {'Global Uncertainty'}")
    print("-" * 135)

    step_records = []

    for idx, task in enumerate(corpus, start=1):
        nom_d, states = harness.get_states_and_nom(task)

        # 1. Select inquiry with functional awareness
        target_organ, chosen_probe, net_eig = planner.select_inquiry(states)
        organ_ref = planner.organs[target_organ]
        l, h = organ_ref.layer, organ_ref.head
        impact_w = organ_ref.compute_functional_impact_weight()

        # 2. Execute
        resp = harness.execute_probe(task, l, h, chosen_probe, nom_d)

        # 3. Update
        planner.record_observation(target_organ, chosen_probe, resp, states[target_organ])

        # 4. Measure global remaining uncertainty
        curr_uncs = [planner.organs[c].get_uncertainty(states[c])["u_total"] for c in planner.organs]
        mean_u = float(np.mean(curr_uncs))

        print(f"#{idx:02d}  | {task['id']:<20} | {target_organ:<10} | {chosen_probe:<8} | {net_eig:8.4f}   | {impact_w:8.3f}   | {resp:+14.4f}   | {mean_u:12.4f}")

        step_records.append({
            "step": idx,
            "task": task["id"],
            "organ": target_organ,
            "probe": chosen_probe,
            "net_eig": net_eig,
            "impact_weight": impact_w,
            "response": resp,
            "global_uncertainty": mean_u
        })

    # -------------------------------------------------------------------------
    # PHASE 2 SPECIALIZATION AUDIT (Steps 7 to 18)
    # -------------------------------------------------------------------------
    print("\n" + "=" * 135)
    print("📊 POST-INITIALIZATION (PHASE 2) TARGETING AUDIT (STEPS 7-18)")
    print("=" * 135)

    phase2_steps = step_records[6:]
    organ_counts = {}
    for r in phase2_steps:
        organ_counts[r["organ"]] = organ_counts.get(r["organ"], 0) + 1

    active_circuit_visits = organ_counts.get("L21H06", 0) + organ_counts.get("L21H01", 0)
    control_heads_visits = organ_counts.get("L17H01", 0) + organ_counts.get("L14H04", 0) + organ_counts.get("L10H00", 0)

    print(f"Phase 2 Total Inquiries: {len(phase2_steps)}")
    for organ_name, count in sorted(organ_counts.items(), key=lambda x: x[1], reverse=True):
        print(f"  • {organ_name:<10}: {count} visits ({count/len(phase2_steps)*100:.1f}%)")

    print("-" * 135)
    print(f"Active Circuit (L21H06 / L21H01) Visits : {active_circuit_visits}")
    print(f"Silent Control Heads Visits             : {control_heads_visits}")

    if active_circuit_visits > control_heads_visits:
        print("👉 VERDICT: 🟢 FUNCTIONAL_ACTIVE_INQUIRY_CONFIRMED (System selectively prioritizes high-impact circuit over inert controls)")
    else:
        print("👉 VERDICT: 🔴 INERT_HEAD_OVER_EXPLORATION")
    print("=" * 135 + "\n")

    with open("m8_2_extended_trace_v2.json", "w", encoding="utf-8") as f:
        json.dump({"milestone": "8.2_V2_FUNCTIONAL_EIG", "trace": step_records}, f, indent=2, ensure_ascii=False)


if __name__ == "__main__":
    run_milestone_8_2_v2()

🔄 Loading Qwen/Qwen2.5-0.5B on CUDA (bfloat16)...


/tmp/ipykernel_2736/2586301128.py:125: DeprecationWarning: HookedTransformer.from_pretrained is deprecated and will be removed in a future major release. Use TransformerBridge.boot_transformers(...) instead, then call enable_compatibility_mode() for HookedTransformer-equivalent numerics. See docs/source/content/migrating_to_v3.md.
  self.model = HookedTransformer.from_pretrained(


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Loaded pretrained model Qwen/Qwen2.5-0.5B into HookedTransformer
✅ Verified token integrity across 18 extended tasks.

🔬 MILESTONE 8.2 (V2): EXTENDED 18-STEP ACTIVE INQUIRY TRACE WITH FUNCTIONAL WEIGHTING
Step  | Task ID              | Organ      | Probe    | Net EIG    | Impact W   | Observed Resp    | Global Uncertainty
---------------------------------------------------------------------------------------------------------------------------------------
#01  | T01_FactParis        | L21H06     | P1       |   0.8600   |    1.000   |        +0.1250   |       0.8003
#02  | T02_BindingBob       | L21H01     | P1       |   0.8600   |    1.000   |        +0.0000   |       0.8060
#03  | T03_DistractorRome   | L17H01     | P1       |   0.8600   |    1.000   |        -0.0625   |       0.7300
#04  | T04_FactTokyo        | L14H04     | P1       |   0.8600   |    1.000   |        +0.0000   |       0.7558
#05  | T05_BindingMary      | L10H00     | P1       |   0.8600   |    1.000   |        -0.12

In [6]:
"""
m8_1_saliency_grounded_efe.py
==============================
Milestone 8.1: Saliency-Grounded Expected Free Energy (EFE) Probing Engine.

Core Mechanism:
  - Grounds epistemic uncertainty with Direct Logit Attribution (DLA):
      DLA(z_C) = (z_C @ W_O) @ (W_U[:, correct] - W_U[:, incorrect])
      Saliency(C) = |DLA(z_C)|
  - Expected Free Energy (EFE):
      EFE(C, P) = (Saliency(C) ** 0.8) * U_total(C) * Diminish(P) - Cost(P)
  - Objective: Automatically prune silent control heads and allocate >85%
    of active inquiries to the decision-relevant circuit (L21H06 / L21H01).
"""

import torch
import numpy as np
import json
from dataclasses import dataclass, field
from typing import Dict, List, Tuple
from transformer_lens import HookedTransformer

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)


# ==============================================================================
# 1. SALIENCY-GROUNDED EPISTEMIC ORGAN
# ==============================================================================

@dataclass
class SaliencyEpistemicOrgan:
    component_name: str
    layer: int
    head: int
    evidence_contexts: List[List[float]] = field(default_factory=list)
    dla_history: List[float] = field(default_factory=list)
    probe_response_history: Dict[str, List[float]] = field(
        default_factory=lambda: {"P1": [], "P2": [], "P3": [], "P4": []}
    )

    def get_uncertainty(self, ctx_tensor: torch.Tensor) -> Dict[str, float]:
        ctx_np = ctx_tensor.detach().cpu().to(torch.float32).numpy()

        if not self.evidence_contexts:
            u_epistemic = 1.0
            u_novelty = 1.0
        else:
            sims = [
                float(np.dot(ctx_np, ev) / (np.linalg.norm(ctx_np) * np.linalg.norm(ev) + 1e-8))
                for ev in self.evidence_contexts
            ]
            effective_evidence = sum(max(0.0, s) for s in sims)
            u_epistemic = float(np.exp(-0.45 * effective_evidence))
            u_novelty = float(max(0.0, 1.0 - max(sims)))

        all_resps = [r for sub in self.probe_response_history.values() for r in sub]
        if len(all_resps) >= 2:
            u_disagree = float(min(1.0, np.var(all_resps) * 4.0))
        else:
            u_disagree = 0.50

        u_total = float(0.40 * u_epistemic + 0.35 * u_novelty + 0.25 * u_disagree)
        return {"u_total": u_total, "u_epistemic": u_epistemic, "u_novelty": u_novelty, "u_disagree": u_disagree}

    def compute_saliency(self, current_dla: float) -> float:
        """Computes behavioral saliency from current DLA and historical attribution magnitude."""
        self.dla_history.append(abs(current_dla))
        recent_dla = float(np.mean(self.dla_history[-5:]))

        # Soft-floor saliency: active heads get high values (0.5 - 3.0), silent heads drop to ~0.02
        saliency = float(max(0.02, recent_dla))
        return saliency

    def compute_efe(self, probe_name: str, ctx_tensor: torch.Tensor, current_dla: float) -> float:
        unc = self.get_uncertainty(ctx_tensor)
        saliency = self.compute_saliency(current_dla)
        n_obs = len(self.probe_response_history[probe_name])

        diminish_factor = float(np.exp(-0.55 * n_obs))

        # Saliency-Grounded Free Energy Formulation
        saliency_weight = (saliency ** 0.85)
        raw_efe = saliency_weight * unc["u_total"] * (0.45 * unc["u_novelty"] + 0.55 * diminish_factor)
        return float(raw_efe)


class SaliencyInquiryEngine:
    def __init__(self, tracked_heads: List[Tuple[int, int]]):
        self.organs: Dict[str, SaliencyEpistemicOrgan] = {
            f"L{l:02d}H{h:02d}": SaliencyEpistemicOrgan(component_name=f"L{l:02d}H{h:02d}", layer=l, head=h)
            for (l, h) in tracked_heads
        }
        self.probe_cost = {"P1": 0.02, "P2": 0.02, "P3": 0.04, "P4": 0.02}

    def select_inquiry(self, current_states: Dict[str, torch.Tensor], dlas: Dict[str, float]) -> Tuple[str, str, float]:
        best_organ, best_probe, max_net = None, None, -float("inf")

        for organ_name, organ in self.organs.items():
            ctx = current_states[organ_name]
            dla_val = dlas[organ_name]
            for p_name in ["P1", "P2", "P3", "P4"]:
                efe = organ.compute_efe(p_name, ctx, dla_val)
                net = efe - self.probe_cost[p_name]
                if net > max_net:
                    max_net = net
                    best_organ = organ_name
                    best_probe = p_name

        return best_organ, best_probe, max_net

    def record_observation(self, organ_name: str, probe_name: str, response: float, ctx_tensor: torch.Tensor):
        organ = self.organs[organ_name]
        organ.evidence_contexts.append(ctx_tensor.detach().cpu().to(torch.float32).numpy().tolist())
        organ.probe_response_history[probe_name].append(response)


# ==============================================================================
# 2. MECHANISTIC HARNESS
# ==============================================================================

class M8_1_SaliencyHarness:
    def __init__(self, model_name: str = "Qwen/Qwen2.5-0.5B", device: str = "cuda"):
        self.device = device if torch.cuda.is_available() else "cpu"
        print(f"🔄 Loading {model_name} on {self.device.upper()} (bfloat16)...")
        self.model = HookedTransformer.from_pretrained(
            model_name, device=self.device, torch_dtype=torch.bfloat16, default_padding_side="left"
        )
        self.model.eval()
        self.tracked_heads = [(21, 6), (21, 1), (17, 1), (14, 4), (10, 0)]
        self.d_head = self.model.cfg.d_head
        v_rand = torch.randn(self.d_head, dtype=torch.float32)
        self.v_ortho = (v_rand / v_rand.norm()).to(self.device)

    def compute_diff(self, logits: torch.Tensor, c_id: int, inc_id: int) -> float:
        return float((logits[0, -1, c_id] - logits[0, -1, inc_id]).detach().cpu().item())

    @torch.no_grad()
    def get_states_and_dla(self, task: Dict) -> Tuple[float, Dict[str, torch.Tensor], Dict[str, float]]:
        clean_toks = self.model.to_tokens(task["clean"])
        c_id = int(self.model.to_single_token(task["correct"]))
        inc_id = int(self.model.to_single_token(task["incorrect"]))

        nom_logits, cache = self.model.run_with_cache(clean_toks)
        nom_diff = self.compute_diff(nom_logits, c_id, inc_id)

        w_u_c = self.model.W_U[:, c_id].detach().cpu().to(torch.float32)
        w_u_inc = self.model.W_U[:, inc_id].detach().cpu().to(torch.float32)
        d_u = (w_u_c - w_u_inc)

        states = {}
        dlas = {}
        for (l, h) in self.tracked_heads:
            z = cache[f"blocks.{l}.attn.hook_z"][0, -1, h, :].clone()
            states[f"L{l:02d}H{h:02d}"] = z

            w_o = self.model.blocks[l].attn.W_O[h].detach().cpu().to(torch.float32)
            head_out = torch.matmul(z.detach().cpu().to(torch.float32), w_o)
            dla = float(torch.dot(head_out, d_u).item())
            dlas[f"L{l:02d}H{h:02d}"] = dla

        return nom_diff, states, dlas

    @torch.no_grad()
    def execute_probe(self, task: Dict, layer: int, head: int, probe_name: str, nom_diff: float) -> float:
        clean_toks = self.model.to_tokens(task["clean"])
        c_id = int(self.model.to_single_token(task["correct"]))
        inc_id = int(self.model.to_single_token(task["incorrect"]))

        if probe_name == "P1":
            def hook_fn(act, hook): act[:, -1, head, :] *= 0.90; return act
        elif probe_name == "P2":
            def hook_fn(act, hook): act[:, -1, head, :] *= 1.10; return act
        elif probe_name == "P3":
            def hook_fn(act, hook): act[:, -1, head, :] += 0.06 * act[:, -1, head, :].norm() * self.v_ortho.to(act.dtype); return act
        elif probe_name == "P4":
            def hook_fn(act, hook): act[:, -1, head, :] -= 0.08 * act[:, -1, head, :]; return act
        else:
            raise ValueError(f"Unknown probe: {probe_name}")

        with self.model.hooks(fwd_hooks=[(f"blocks.{layer}.attn.hook_z", hook_fn)]):
            p_logits, _ = self.model.run_with_cache(clean_toks)
        return float(nom_diff - self.compute_diff(p_logits, c_id, inc_id))


# ==============================================================================
# 3. TASK CORPUS (18 EXTENDED MULTI-REGIME TASKS)
# ==============================================================================

def get_extended_corpus():
    return [
        {"id": "T01_FactParis", "clean": "The Eiffel Tower is located in the city of", "correct": " Paris", "incorrect": " Rome"},
        {"id": "T02_BindingBob", "clean": "Alice handed a red apple to Bob. The apple was received by", "correct": " Bob", "incorrect": " Alice"},
        {"id": "T03_DistractorRome", "clean": "Berlin has monuments, yet the ancient Colosseum stands in", "correct": " Rome", "incorrect": " Berlin"},
        {"id": "T04_FactTokyo", "clean": "The capital of Japan is the bustling city of", "correct": " Tokyo", "incorrect": " Cairo"},
        {"id": "T05_BindingMary", "clean": "John sent a private email to Mary. The email reached", "correct": " Mary", "incorrect": " John"},
        {"id": "T06_PyramidsEgypt", "clean": "Despite the fame of Rome, the ancient Pyramids are preserved in", "correct": " Egypt", "incorrect": " Cairo"},
        {"id": "T07_FactLondon", "clean": "The famous Big Ben clock tower is located in", "correct": " London", "incorrect": " York"},
        {"id": "T08_BindingSarah", "clean": "David passed the heavy book to Sarah. The book was taken by", "correct": " Sarah", "incorrect": " David"},
        {"id": "T09_DistractorParis", "clean": "While Madrid is famous for soccer, the capital of France is", "correct": " Paris", "incorrect": " Madrid"},
        {"id": "T10_FactBerlin", "clean": "The official capital city of Germany is", "correct": " Berlin", "incorrect": " Munich"},
        {"id": "T11_BindingEmma", "clean": "Michael offered a cup of tea to Emma. The tea was accepted by", "correct": " Emma", "incorrect": " Michael"},
        {"id": "T12_DistractorAthens", "clean": "Although Tokyo is very large, the ancient Parthenon is located in", "correct": " Athens", "incorrect": " Tokyo"},
        {"id": "T13_FactMoscow", "clean": "The historic Kremlin is located in the city of", "correct": " Moscow", "incorrect": " Athens"},
        {"id": "T14_BindingJack", "clean": "Tom tossed the baseball to Jack. The ball was caught by", "correct": " Jack", "incorrect": " Tom"},
        {"id": "T15_DistractorRome2", "clean": "Even though Cairo has pyramids, the capital of Italy is", "correct": " Rome", "incorrect": " Cairo"},
        {"id": "T16_FactMadrid", "clean": "The city serving as the capital of Spain is definitely", "correct": " Madrid", "incorrect": " Rome"},
        {"id": "T17_BindingLisa", "clean": "Robert wrote a letter to Lisa. The letter was delivered to", "correct": " Lisa", "incorrect": " Robert"},
        {"id": "T18_DistractorBerlin2", "clean": "The official capital of Germany is definitely not Rome, it is", "correct": " Berlin", "incorrect": " Rome"}
    ]


# ==============================================================================
# 4. MASTER EXECUTION & VALIDATION CONTROLLER
# ==============================================================================

def run_milestone_8_1():
    harness = M8_1_SaliencyHarness()
    planner = SaliencyInquiryEngine(harness.tracked_heads)
    corpus = get_extended_corpus()

    for t in corpus:
        _ = harness.model.to_single_token(t["correct"])
        _ = harness.model.to_single_token(t["incorrect"])
    print(f"✅ Verified token integrity across {len(corpus)} evaluation tasks.")

    print("\n" + "=" * 135)
    print("🔬 MILESTONE 8.1: SALIENCY-GROUNDED EXPECTED FREE ENERGY (EFE) TRACE")
    print("=" * 135)
    print(f"{'Step':<5} | {'Task ID':<22} | {'Organ':<10} | {'Probe':<8} | {'Net EFE':<10} | {'DLA Magnitude':<15} | {'Observed Resp':<15}")
    print("-" * 135)

    step_records = []

    for idx, task in enumerate(corpus, start=1):
        nom_d, states, dlas = harness.get_states_and_dla(task)

        target_organ, chosen_probe, net_efe = planner.select_inquiry(states, dlas)
        organ_ref = planner.organs[target_organ]
        l, h = organ_ref.layer, organ_ref.head

        resp = harness.execute_probe(task, l, h, chosen_probe, nom_d)
        planner.record_observation(target_organ, chosen_probe, resp, states[target_organ])

        current_dla = dlas[target_organ]
        print(f"#{idx:02d}  | {task['id']:<22} | {target_organ:<10} | {chosen_probe:<8} | {net_efe:8.4f}   | {abs(current_dla):13.4f}   | {resp:+13.4f}")

        step_records.append({
            "step": idx,
            "task": task["id"],
            "organ": target_organ,
            "probe": chosen_probe,
            "net_efe": net_efe,
            "dla": current_dla,
            "response": resp
        })

    # -------------------------------------------------------------------------
    # CIRCUIT PRUNING & CONCENTRATION AUDIT
    # -------------------------------------------------------------------------
    print("\n" + "=" * 135)
    print("📊 INQUIRY ALLOCATION & AUTOMATIC CIRCUIT PRUNING AUDIT")
    print("=" * 135)

    organ_counts = {}
    for r in step_records:
        organ_counts[r["organ"]] = organ_counts.get(r["organ"], 0) + 1

    active_circuit_visits = organ_counts.get("L21H06", 0) + organ_counts.get("L21H01", 0)
    control_heads_visits = organ_counts.get("L17H01", 0) + organ_counts.get("L14H04", 0) + organ_counts.get("L10H00", 0)
    total_steps = len(step_records)
    active_circuit_pct = (active_circuit_visits / total_steps) * 100.0

    print(f"Total Probing Actions Executed: {total_steps}")
    for organ_name, count in sorted(organ_counts.items(), key=lambda x: x[1], reverse=True):
        print(f"  • {organ_name:<10}: {count:2d} visits ({count/total_steps*100:5.1f}%)")

    print("-" * 135)
    print(f"Active Decision Circuit (L21H06 / L21H01) : {active_circuit_visits:2d} / {total_steps} ({active_circuit_pct:.1f}%)")
    print(f"Silent Control Components (L17, L14, L10)  : {control_heads_visits:2d} / {total_steps} ({100 - active_circuit_pct:.1f}%)")

    if active_circuit_pct >= 80.0:
        print("👉 AUDIT VERDICT: 🟢 SALIENCY_GROUNDED_PRUNING_VALIDATED (Silent control heads successfully pruned from inquiry tree)")
    else:
        print("👉 AUDIT VERDICT: 🔴 INSUFFICIENT_CIRCUIT_CONCENTRATION")
    print("=" * 135 + "\n")

    with open("m8_1_saliency_grounded_trace.json", "w", encoding="utf-8") as f:
        json.dump({
            "milestone": "8.1_SALIENCY_GROUNDED_EFE",
            "active_circuit_pct": active_circuit_pct,
            "trace": step_records
        }, f, indent=2, ensure_ascii=False)
    print("📁 Comprehensive audit telemetry saved to: m8_1_saliency_grounded_trace.json")


if __name__ == "__main__":
    run_milestone_8_1()

🔄 Loading Qwen/Qwen2.5-0.5B on CUDA (bfloat16)...


/tmp/ipykernel_2736/328808897.py:127: DeprecationWarning: HookedTransformer.from_pretrained is deprecated and will be removed in a future major release. Use TransformerBridge.boot_transformers(...) instead, then call enable_compatibility_mode() for HookedTransformer-equivalent numerics. See docs/source/content/migrating_to_v3.md.
  self.model = HookedTransformer.from_pretrained(


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Loaded pretrained model Qwen/Qwen2.5-0.5B into HookedTransformer
✅ Verified token integrity across 18 evaluation tasks.

🔬 MILESTONE 8.1: SALIENCY-GROUNDED EXPECTED FREE ENERGY (EFE) TRACE
Step  | Task ID                | Organ      | Probe    | Net EFE    | DLA Magnitude   | Observed Resp  
---------------------------------------------------------------------------------------------------------------------------------------
#01  | T01_FactParis          | L21H06     | P1       |   6.0918   |        9.8431   |       +0.1250
#02  | T02_BindingBob         | L21H06     | P2       |   4.2184   |        0.0080   |       +0.0000
#03  | T03_DistractorRome     | L21H06     | P4       |   1.1498   |        3.2950   |       -0.1250
#04  | T04_FactTokyo          | L21H06     | P3       |   3.5763   |       11.0782   |       +0.0000
#05  | T05_BindingMary        | L21H01     | P1       |   2.7983   |        0.9068   |       -0.1250
#06  | T06_PyramidsEgypt      | L21H01     | P4       |   0.5715  

In [7]:
"""
m8_1_label_free_saliency_efe.py
================================
Milestone 8.1 (Label-Free): Autonomous Multi-Organ Probing via Unsupervised Saliency.

Key Properties:
  - ZERO Label Leakage: No access to task["correct"] or task["incorrect"] during state assessment.
  - Unsupervised Saliency: Evaluates head attribution on the model's own Top-1 vs Top-2 logit margin.
  - Saliency-Grounded EFE: Balances epistemic uncertainty with intrinsic competitive attribution.
"""

import torch
import numpy as np
import json
from dataclasses import dataclass, field
from typing import Dict, List, Tuple
from transformer_lens import HookedTransformer

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)


# ==============================================================================
# 1. LABEL-FREE SALIENCY-GROUNDED ORGAN
# ==============================================================================

@dataclass
class LabelFreeEpistemicOrgan:
    component_name: str
    layer: int
    head: int
    evidence_contexts: List[List[float]] = field(default_factory=list)
    saliency_history: List[float] = field(default_factory=list)
    probe_response_history: Dict[str, List[float]] = field(
        default_factory=lambda: {"P1": [], "P2": [], "P3": [], "P4": []}
    )

    def get_uncertainty(self, ctx_tensor: torch.Tensor) -> Dict[str, float]:
        ctx_np = ctx_tensor.detach().cpu().to(torch.float32).numpy()

        if not self.evidence_contexts:
            u_epistemic = 1.0
            u_novelty = 1.0
        else:
            sims = [
                float(np.dot(ctx_np, ev) / (np.linalg.norm(ctx_np) * np.linalg.norm(ev) + 1e-8))
                for ev in self.evidence_contexts
            ]
            effective_evidence = sum(max(0.0, s) for s in sims)
            u_epistemic = float(np.exp(-0.45 * effective_evidence))
            u_novelty = float(max(0.0, 1.0 - max(sims)))

        all_resps = [r for sub in self.probe_response_history.values() for r in sub]
        if len(all_resps) >= 2:
            u_disagree = float(min(1.0, np.var(all_resps) * 4.0))
        else:
            u_disagree = 0.50

        u_total = float(0.40 * u_epistemic + 0.35 * u_novelty + 0.25 * u_disagree)
        return {"u_total": u_total, "u_epistemic": u_epistemic, "u_novelty": u_novelty, "u_disagree": u_disagree}

    def compute_saliency_weight(self, current_unsupervised_saliency: float) -> float:
        self.saliency_history.append(current_unsupervised_saliency)
        recent_sal = float(np.mean(self.saliency_history[-5:]))
        # Soft-floor ensures silent heads drop to ~0.05 while active heads stay high
        return float(max(0.05, recent_sal))

    def compute_efe(self, probe_name: str, ctx_tensor: torch.Tensor, current_unsupervised_saliency: float) -> float:
        unc = self.get_uncertainty(ctx_tensor)
        sal_weight = self.compute_saliency_weight(current_unsupervised_saliency)
        n_obs = len(self.probe_response_history[probe_name])

        diminish_factor = float(np.exp(-0.55 * n_obs))

        # Saliency-Grounded Free Energy (No Label Dependency)
        raw_efe = (sal_weight ** 0.85) * unc["u_total"] * (0.45 * unc["u_novelty"] + 0.55 * diminish_factor)
        return float(raw_efe)


class AutonomousInquiryPlanner:
    def __init__(self, tracked_heads: List[Tuple[int, int]]):
        self.organs: Dict[str, LabelFreeEpistemicOrgan] = {
            f"L{l:02d}H{h:02d}": LabelFreeEpistemicOrgan(component_name=f"L{l:02d}H{h:02d}", layer=l, head=h)
            for (l, h) in tracked_heads
        }
        self.probe_cost = {"P1": 0.02, "P2": 0.02, "P3": 0.04, "P4": 0.02}

    def select_inquiry(self, current_states: Dict[str, torch.Tensor], saliencies: Dict[str, float]) -> Tuple[str, str, float]:
        best_organ, best_probe, max_net = None, None, -float("inf")

        for organ_name, organ in self.organs.items():
            ctx = current_states[organ_name]
            sal_val = saliencies[organ_name]
            for p_name in ["P1", "P2", "P3", "P4"]:
                efe = organ.compute_efe(p_name, ctx, sal_val)
                net = efe - self.probe_cost[p_name]
                if net > max_net:
                    max_net = net
                    best_organ = organ_name
                    best_probe = p_name

        return best_organ, best_probe, max_net

    def record_observation(self, organ_name: str, probe_name: str, response: float, ctx_tensor: torch.Tensor):
        organ = self.organs[organ_name]
        organ.evidence_contexts.append(ctx_tensor.detach().cpu().to(torch.float32).numpy().tolist())
        organ.probe_response_history[probe_name].append(response)


# ==============================================================================
# 2. MECHANISTIC HARNESS (LABEL-FREE INFERENCE)
# ==============================================================================

class M8_1_LabelFreeHarness:
    def __init__(self, model_name: str = "Qwen/Qwen2.5-0.5B", device: str = "cuda"):
        self.device = device if torch.cuda.is_available() else "cpu"
        print(f"🔄 Loading {model_name} on {self.device.upper()} (bfloat16)...")
        self.model = HookedTransformer.from_pretrained(
            model_name, device=self.device, torch_dtype=torch.bfloat16, default_padding_side="left"
        )
        self.model.eval()
        self.tracked_heads = [(21, 6), (21, 1), (17, 1), (14, 4), (10, 0)]
        self.d_head = self.model.cfg.d_head
        v_rand = torch.randn(self.d_head, dtype=torch.float32)
        self.v_ortho = (v_rand / v_rand.norm()).to(self.device)

    @torch.no_grad()
    def extract_unsupervised_states_and_saliency(self, prompt: str) -> Tuple[int, int, Dict[str, torch.Tensor], Dict[str, float]]:
        """Extracts states and saliencies based ONLY on the model's internal Top-1 vs Top-2 prediction."""
        tokens = self.model.to_tokens(prompt)
        nom_logits, cache = self.model.run_with_cache(tokens)

        last_token_logits = nom_logits[0, -1, :]
        top_tokens = torch.topk(last_token_logits, k=2).indices
        t1, t2 = int(top_tokens[0].item()), int(top_tokens[1].item())

        w_u_t1 = self.model.W_U[:, t1].detach().cpu().to(torch.float32)
        w_u_t2 = self.model.W_U[:, t2].detach().cpu().to(torch.float32)
        d_u_competitive = (w_u_t1 - w_u_t2)

        states = {}
        saliencies = {}

        for (l, h) in self.tracked_heads:
            z = cache[f"blocks.{l}.attn.hook_z"][0, -1, h, :].clone()
            states[f"L{l:02d}H{h:02d}"] = z

            w_o = self.model.blocks[l].attn.W_O[h].detach().cpu().to(torch.float32)
            head_out = torch.matmul(z.detach().cpu().to(torch.float32), w_o)

            # Intrinsic Saliency = Competitive logit projection + vector norm contribution
            competitive_dla = float(torch.dot(head_out, d_u_competitive).item())
            out_norm = float(head_out.norm().item())

            # Combined unsupervised saliency metric
            saliency = abs(competitive_dla) + 0.15 * out_norm
            saliencies[f"L{l:02d}H{h:02d}"] = saliency

        return t1, t2, states, saliencies

    @torch.no_grad()
    def execute_probe(self, prompt: str, layer: int, head: int, probe_name: str, t1: int, t2: int) -> float:
        """Measures probe impact on the model's own competitive margin (t1 vs t2)."""
        tokens = self.model.to_tokens(prompt)
        nom_logits = self.model(tokens)
        nom_margin = float((nom_logits[0, -1, t1] - nom_logits[0, -1, t2]).item())

        if probe_name == "P1":
            def hook_fn(act, hook): act[:, -1, head, :] *= 0.90; return act
        elif probe_name == "P2":
            def hook_fn(act, hook): act[:, -1, head, :] *= 1.10; return act
        elif probe_name == "P3":
            def hook_fn(act, hook): act[:, -1, head, :] += 0.06 * act[:, -1, head, :].norm() * self.v_ortho.to(act.dtype); return act
        elif probe_name == "P4":
            def hook_fn(act, hook): act[:, -1, head, :] -= 0.08 * act[:, -1, head, :]; return act
        else:
            raise ValueError(f"Unknown probe: {probe_name}")

        with self.model.hooks(fwd_hooks=[(f"blocks.{layer}.attn.hook_z", hook_fn)]):
            p_logits = self.model(tokens)
        p_margin = float((p_logits[0, -1, t1] - p_logits[0, -1, t2]).item())

        return float(nom_margin - p_margin)


# ==============================================================================
# 3. TASK CORPUS (18 TASKS EVALUATED UNLABELED)
# ==============================================================================

def get_prompts():
    return [
        {"id": "T01_FactParis", "prompt": "The Eiffel Tower is located in the city of"},
        {"id": "T02_BindingBob", "prompt": "Alice handed a red apple to Bob. The apple was received by"},
        {"id": "T03_DistractorRome", "prompt": "Berlin has monuments, yet the ancient Colosseum stands in"},
        {"id": "T04_FactTokyo", "prompt": "The capital of Japan is the bustling city of"},
        {"id": "T05_BindingMary", "prompt": "John sent a private email to Mary. The email reached"},
        {"id": "T06_PyramidsEgypt", "prompt": "Despite the fame of Rome, the ancient Pyramids are preserved in"},
        {"id": "T07_FactLondon", "prompt": "The famous Big Ben clock tower is located in"},
        {"id": "T08_BindingSarah", "prompt": "David passed the heavy book to Sarah. The book was taken by"},
        {"id": "T09_DistractorParis", "prompt": "While Madrid is famous for soccer, the capital of France is"},
        {"id": "T10_FactBerlin", "prompt": "The official capital city of Germany is"},
        {"id": "T11_BindingEmma", "prompt": "Michael offered a cup of tea to Emma. The tea was accepted by"},
        {"id": "T12_DistractorAthens", "prompt": "Although Tokyo is very large, the ancient Parthenon is located in"},
        {"id": "T13_FactMoscow", "prompt": "The historic Kremlin is located in the city of"},
        {"id": "T14_BindingJack", "prompt": "Tom tossed the baseball to Jack. The ball was caught by"},
        {"id": "T15_DistractorRome2", "prompt": "Even though Cairo has pyramids, the capital of Italy is"},
        {"id": "T16_FactMadrid", "prompt": "The city serving as the capital of Spain is definitely"},
        {"id": "T17_BindingLisa", "prompt": "Robert wrote a letter to Lisa. The letter was delivered to"},
        {"id": "T18_DistractorBerlin2", "prompt": "The official capital of Germany is definitely not Rome, it is"}
    ]


# ==============================================================================
# 4. MASTER EXPERIMENTAL RUNNER
# ==============================================================================

def run_milestone_8_1_label_free():
    harness = M8_1_LabelFreeHarness()
    planner = AutonomousInquiryPlanner(harness.tracked_heads)
    prompts = get_prompts()

    print("\n" + "=" * 135)
    print("🔬 MILESTONE 8.1 (LABEL-FREE): AUTONOMOUS EFE PROBING TRACE (TOP-1 vs TOP-2 SALIENCY)")
    print("=" * 135)
    print(f"{'Step':<5} | {'Task ID':<22} | {'Organ':<10} | {'Probe':<8} | {'Net EFE':<10} | {'Unsupervised Sal':<18} | {'Observed Resp':<15}")
    print("-" * 135)

    step_records = []

    for idx, item in enumerate(prompts, start=1):
        prompt_text = item["prompt"]

        # 1. Autonomous state extraction (ZERO ground truth labels used)
        t1, t2, states, saliencies = harness.extract_unsupervised_states_and_saliency(prompt_text)

        # 2. Select optimal inquiry via unsupervised EFE
        target_organ, chosen_probe, net_efe = planner.select_inquiry(states, saliencies)
        organ_ref = planner.organs[target_organ]
        l, h = organ_ref.layer, organ_ref.head

        # 3. Execute probe on internal competitive margin
        resp = harness.execute_probe(prompt_text, l, h, chosen_probe, t1, t2)
        planner.record_observation(target_organ, chosen_probe, resp, states[target_organ])

        cur_sal = saliencies[target_organ]
        print(f"#{idx:02d}  | {item['id']:<22} | {target_organ:<10} | {chosen_probe:<8} | {net_efe:8.4f}   | {cur_sal:16.4f}   | {resp:+13.4f}")

        step_records.append({
            "step": idx,
            "task": item["id"],
            "organ": target_organ,
            "probe": chosen_probe,
            "net_efe": net_efe,
            "unsupervised_saliency": cur_sal,
            "response": resp
        })

    # -------------------------------------------------------------------------
    # CIRCUIT ALLOCATION AUDIT
    # -------------------------------------------------------------------------
    print("\n" + "=" * 135)
    print("📊 LABEL-FREE INQUIRY ALLOCATION AUDIT")
    print("=" * 135)

    organ_counts = {}
    for r in step_records:
        organ_counts[r["organ"]] = organ_counts.get(r["organ"], 0) + 1

    active_circuit_visits = organ_counts.get("L21H06", 0) + organ_counts.get("L21H01", 0)
    control_heads_visits = organ_counts.get("L17H01", 0) + organ_counts.get("L14H04", 0) + organ_counts.get("L10H00", 0)
    total_steps = len(step_records)
    active_circuit_pct = (active_circuit_visits / total_steps) * 100.0

    print(f"Total Probing Actions Executed: {total_steps}")
    for organ_name, count in sorted(organ_counts.items(), key=lambda x: x[1], reverse=True):
        print(f"  • {organ_name:<10}: {count:2d} visits ({count/total_steps*100:5.1f}%)")

    print("-" * 135)
    print(f"Active Decision Circuit (L21H06 / L21H01) : {active_circuit_visits:2d} / {total_steps} ({active_circuit_pct:.1f}%)")
    print(f"Silent Control Components (L17, L14, L10)  : {control_heads_visits:2d} / {total_steps} ({100 - active_circuit_pct:.1f}%)")

    if active_circuit_pct >= 75.0:
        print("👉 AUDIT VERDICT: 🟢 LABEL_FREE_ACTIVE_INQUIRY_VALIDATED (Autonomous circuit targeting without label leakage)")
    else:
        print("👉 AUDIT VERDICT: 🔴 INERT_HEAD_OVER_EXPLORATION")
    print("=" * 135 + "\n")

    with open("m8_1_label_free_trace.json", "w", encoding="utf-8") as f:
        json.dump({
            "milestone": "8.1_LABEL_FREE_EFE",
            "active_circuit_pct": active_circuit_pct,
            "trace": step_records
        }, f, indent=2, ensure_ascii=False)
    print("📁 Telemetry saved to: m8_1_label_free_trace.json")


if __name__ == "__main__":
    run_milestone_8_1_label_free()

🔄 Loading Qwen/Qwen2.5-0.5B on CUDA (bfloat16)...


/tmp/ipykernel_2736/952031148.py:119: DeprecationWarning: HookedTransformer.from_pretrained is deprecated and will be removed in a future major release. Use TransformerBridge.boot_transformers(...) instead, then call enable_compatibility_mode() for HookedTransformer-equivalent numerics. See docs/source/content/migrating_to_v3.md.
  self.model = HookedTransformer.from_pretrained(


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Loaded pretrained model Qwen/Qwen2.5-0.5B into HookedTransformer

🔬 MILESTONE 8.1 (LABEL-FREE): AUTONOMOUS EFE PROBING TRACE (TOP-1 vs TOP-2 SALIENCY)
Step  | Task ID                | Organ      | Probe    | Net EFE    | Unsupervised Sal   | Observed Resp  
---------------------------------------------------------------------------------------------------------------------------------------
#01  | T01_FactParis          | L21H06     | P1       |   5.8816   |           9.4461   |       +0.0000
#02  | T02_BindingBob         | L21H06     | P2       |   4.1238   |           0.2164   |       +0.0000
#03  | T03_DistractorRome     | L21H06     | P4       |   0.5028   |           1.2340   |       +0.1250
#04  | T04_FactTokyo          | L21H06     | P3       |   2.0961   |           6.4122   |       +0.0000
#05  | T05_BindingMary        | L21H01     | P1       |   2.0630   |           1.1881   |       -0.1250
#06  | T06_PyramidsEgypt      | L21H06     | P1       |   0.7744   |           1.3018 

In [8]:
"""
m8_1_multiseed_robustness_audit.py
===================================
Milestone 8.1: Multi-Seed Statistical Validation of Label-Free Active Probing.

Runs across 3 independent random seeds:
  - Randomizes prompt sequence presentation order.
  - Re-generates random orthogonal probe vectors.
  - Measures Active Circuit Allocation Rate (Mean ± Std).
"""

import torch
import numpy as np
import json
import random
from dataclasses import dataclass, field
from typing import Dict, List, Tuple
from transformer_lens import HookedTransformer


# ==============================================================================
# 1. CORE ENGINES
# ==============================================================================

@dataclass
class LabelFreeEpistemicOrgan:
    component_name: str
    layer: int
    head: int
    evidence_contexts: List[List[float]] = field(default_factory=list)
    saliency_history: List[float] = field(default_factory=list)
    probe_response_history: Dict[str, List[float]] = field(
        default_factory=lambda: {"P1": [], "P2": [], "P3": [], "P4": []}
    )

    def get_uncertainty(self, ctx_tensor: torch.Tensor) -> Dict[str, float]:
        ctx_np = ctx_tensor.detach().cpu().to(torch.float32).numpy()
        if not self.evidence_contexts:
            u_epistemic = 1.0
            u_novelty = 1.0
        else:
            sims = [
                float(np.dot(ctx_np, ev) / (np.linalg.norm(ctx_np) * np.linalg.norm(ev) + 1e-8))
                for ev in self.evidence_contexts
            ]
            effective_evidence = sum(max(0.0, s) for s in sims)
            u_epistemic = float(np.exp(-0.45 * effective_evidence))
            u_novelty = float(max(0.0, 1.0 - max(sims)))

        all_resps = [r for sub in self.probe_response_history.values() for r in sub]
        u_disagree = float(min(1.0, np.var(all_resps) * 4.0)) if len(all_resps) >= 2 else 0.50
        u_total = float(0.40 * u_epistemic + 0.35 * u_novelty + 0.25 * u_disagree)
        return {"u_total": u_total, "u_epistemic": u_epistemic, "u_novelty": u_novelty, "u_disagree": u_disagree}

    def compute_efe(self, probe_name: str, ctx_tensor: torch.Tensor, current_saliency: float) -> float:
        self.saliency_history.append(current_saliency)
        sal_weight = float(max(0.05, np.mean(self.saliency_history[-5:])))
        unc = self.get_uncertainty(ctx_tensor)
        n_obs = len(self.probe_response_history[probe_name])
        diminish_factor = float(np.exp(-0.55 * n_obs))
        return float((sal_weight ** 0.85) * unc["u_total"] * (0.45 * unc["u_novelty"] + 0.55 * diminish_factor))


class AutonomousPlanner:
    def __init__(self, tracked_heads: List[Tuple[int, int]]):
        self.organs = {
            f"L{l:02d}H{h:02d}": LabelFreeEpistemicOrgan(component_name=f"L{l:02d}H{h:02d}", layer=l, head=h)
            for (l, h) in tracked_heads
        }
        self.probe_cost = {"P1": 0.02, "P2": 0.02, "P3": 0.04, "P4": 0.02}

    def select_inquiry(self, states: Dict[str, torch.Tensor], saliencies: Dict[str, float]) -> Tuple[str, str, float]:
        best_organ, best_probe, max_net = None, None, -float("inf")
        for organ_name, organ in self.organs.items():
            ctx, sal = states[organ_name], saliencies[organ_name]
            for p_name in ["P1", "P2", "P3", "P4"]:
                net = organ.compute_efe(p_name, ctx, sal) - self.probe_cost[p_name]
                if net > max_net:
                    max_net, best_organ, best_probe = net, organ_name, p_name
        return best_organ, best_probe, max_net

    def record(self, organ_name: str, probe_name: str, response: float, ctx_tensor: torch.Tensor):
        organ = self.organs[organ_name]
        organ.evidence_contexts.append(ctx_tensor.detach().cpu().to(torch.float32).numpy().tolist())
        organ.probe_response_history[probe_name].append(response)


# ==============================================================================
# 2. HARNESS & DATA
# ==============================================================================

class MultiSeedHarness:
    def __init__(self, model_name: str = "Qwen/Qwen2.5-0.5B", device: str = "cuda"):
        self.device = device if torch.cuda.is_available() else "cpu"
        print(f"🔄 Loading {model_name} on {self.device.upper()} (bfloat16)...")
        self.model = HookedTransformer.from_pretrained(
            model_name, device=self.device, torch_dtype=torch.bfloat16, default_padding_side="left"
        )
        self.model.eval()
        self.tracked_heads = [(21, 6), (21, 1), (17, 1), (14, 4), (10, 0)]
        self.d_head = self.model.cfg.d_head

    def set_seed(self, seed: int):
        torch.manual_seed(seed)
        np.random.seed(seed)
        random.seed(seed)
        v_rand = torch.randn(self.d_head, dtype=torch.float32)
        self.v_ortho = (v_rand / v_rand.norm()).to(self.device)

    @torch.no_grad()
    def get_states_and_saliency(self, prompt: str) -> Tuple[int, int, Dict[str, torch.Tensor], Dict[str, float]]:
        tokens = self.model.to_tokens(prompt)
        nom_logits, cache = self.model.run_with_cache(tokens)
        top_tokens = torch.topk(nom_logits[0, -1, :], k=2).indices
        t1, t2 = int(top_tokens[0].item()), int(top_tokens[1].item())

        d_u = (self.model.W_U[:, t1] - self.model.W_U[:, t2]).detach().cpu().to(torch.float32)
        states, saliencies = {}, {}

        for (l, h) in self.tracked_heads:
            z = cache[f"blocks.{l}.attn.hook_z"][0, -1, h, :].clone()
            states[f"L{l:02d}H{h:02d}"] = z
            w_o = self.model.blocks[l].attn.W_O[h].detach().cpu().to(torch.float32)
            head_out = torch.matmul(z.detach().cpu().to(torch.float32), w_o)
            saliency = abs(float(torch.dot(head_out, d_u).item())) + 0.15 * float(head_out.norm().item())
            saliencies[f"L{l:02d}H{h:02d}"] = saliency

        return t1, t2, states, saliencies

    @torch.no_grad()
    def execute_probe(self, prompt: str, layer: int, head: int, probe_name: str, t1: int, t2: int) -> float:
        tokens = self.model.to_tokens(prompt)
        nom_margin = float((self.model(tokens)[0, -1, t1] - self.model(tokens)[0, -1, t2]).item())

        if probe_name == "P1":
            def hook_fn(act, hook): act[:, -1, head, :] *= 0.90; return act
        elif probe_name == "P2":
            def hook_fn(act, hook): act[:, -1, head, :] *= 1.10; return act
        elif probe_name == "P3":
            def hook_fn(act, hook): act[:, -1, head, :] += 0.06 * act[:, -1, head, :].norm() * self.v_ortho.to(act.dtype); return act
        elif probe_name == "P4":
            def hook_fn(act, hook): act[:, -1, head, :] -= 0.08 * act[:, -1, head, :]; return act

        with self.model.hooks(fwd_hooks=[(f"blocks.{layer}.attn.hook_z", hook_fn)]):
            p_logits = self.model(tokens)
        p_margin = float((p_logits[0, -1, t1] - p_logits[0, -1, t2]).item())
        return float(nom_margin - p_margin)


def get_base_prompts():
    return [
        {"id": "T01_FactParis", "prompt": "The Eiffel Tower is located in the city of"},
        {"id": "T02_BindingBob", "prompt": "Alice handed a red apple to Bob. The apple was received by"},
        {"id": "T03_DistractorRome", "prompt": "Berlin has monuments, yet the ancient Colosseum stands in"},
        {"id": "T04_FactTokyo", "prompt": "The capital of Japan is the bustling city of"},
        {"id": "T05_BindingMary", "prompt": "John sent a private email to Mary. The email reached"},
        {"id": "T06_PyramidsEgypt", "prompt": "Despite the fame of Rome, the ancient Pyramids are preserved in"},
        {"id": "T07_FactLondon", "prompt": "The famous Big Ben clock tower is located in"},
        {"id": "T08_BindingSarah", "prompt": "David passed the heavy book to Sarah. The book was taken by"},
        {"id": "T09_DistractorParis", "prompt": "While Madrid is famous for soccer, the capital of France is"},
        {"id": "T10_FactBerlin", "prompt": "The official capital city of Germany is"},
        {"id": "T11_BindingEmma", "prompt": "Michael offered a cup of tea to Emma. The tea was accepted by"},
        {"id": "T12_DistractorAthens", "prompt": "Although Tokyo is very large, the ancient Parthenon is located in"},
        {"id": "T13_FactMoscow", "prompt": "The historic Kremlin is located in the city of"},
        {"id": "T14_BindingJack", "prompt": "Tom tossed the baseball to Jack. The ball was caught by"},
        {"id": "T15_DistractorRome2", "prompt": "Even though Cairo has pyramids, the capital of Italy is"},
        {"id": "T16_FactMadrid", "prompt": "The city serving as the capital of Spain is definitely"},
        {"id": "T17_BindingLisa", "prompt": "Robert wrote a letter to Lisa. The letter was delivered to"},
        {"id": "T18_DistractorBerlin2", "prompt": "The official capital of Germany is definitely not Rome, it is"}
    ]


# ==============================================================================
# 3. MULTI-SEED RUNNER
# ==============================================================================

def run_multiseed_audit():
    seeds = [42, 137, 999]
    harness = MultiSeedHarness()
    base_prompts = get_base_prompts()

    seed_results = {}

    print("\n" + "=" * 110)
    print("🔬 MILESTONE 8.1: MULTI-SEED ROBUSTNESS AUDIT (LABEL-FREE ACTIVE PROBING)")
    print("=" * 110)

    for s in seeds:
        harness.set_seed(s)
        planner = AutonomousPlanner(harness.tracked_heads)

        # Shuffle task presentation order based on seed
        prompts = base_prompts.copy()
        random.shuffle(prompts)

        organ_counts = {f"L{l:02d}H{h:02d}": 0 for (l, h) in harness.tracked_heads}

        for item in prompts:
            t1, t2, states, saliencies = harness.get_states_and_saliency(item["prompt"])
            target_organ, chosen_probe, _ = planner.select_inquiry(states, saliencies)
            l, h = planner.organs[target_organ].layer, planner.organs[target_organ].head
            resp = harness.execute_probe(item["prompt"], l, h, chosen_probe, t1, t2)
            planner.record(target_organ, chosen_probe, resp, states[target_organ])
            organ_counts[target_organ] += 1

        active_circuit_count = organ_counts["L21H06"] + organ_counts["L21H01"]
        active_pct = (active_circuit_count / len(prompts)) * 100.0

        seed_results[s] = {
            "counts": organ_counts,
            "active_circuit_count": active_circuit_count,
            "active_pct": active_pct
        }

        print(f"Seed {s:4d} | L21H06: {organ_counts['L21H06']:2d} | L21H01: {organ_counts['L21H01']:2d} | Controls (L17/L14/L10): {18-active_circuit_count:2d} | Active Circuit: {active_pct:5.1f}%")

    # -------------------------------------------------------------------------
    # STATISTICAL SUMMARY
    # -------------------------------------------------------------------------
    print("=" * 110)
    pcts = [data["active_pct"] for data in seed_results.values()]
    mean_pct = float(np.mean(pcts))
    std_pct = float(np.std(pcts))

    print(f"\n📊 AGGREGATE METRICS ACROSS {len(seeds)} INDEPENDENT SEEDS:")
    print(f"  • Mean Active Circuit Allocation Rate : {mean_pct:.2f}% ± {std_pct:.2f}%")
    print(f"  • Individual Seed Rates                : {[f'{p:.1f}%' for p in pcts]}")
    print("-" * 110)

    if mean_pct >= 75.0 and (mean_pct - std_pct) >= 70.0:
        print("👉 MULTI-SEED AUDIT VERDICT: 🟢 ROBUST_LABEL_FREE_DISCOVERY_VALIDATED")
        print("   (Active circuit prioritization is statistically stable across permutations and orthogonal bases)")
    else:
        print("👉 MULTI-SEED AUDIT VERDICT: 🔴 SENSITIVE_TO_SEED_NOISE")
    print("=" * 110 + "\n")

    with open("m8_1_multiseed_audit.json", "w", encoding="utf-8") as f:
        json.dump({"seeds": seed_results, "mean_pct": mean_pct, "std_pct": std_pct}, f, indent=2)


if __name__ == "__main__":
    run_multiseed_audit()

🔄 Loading Qwen/Qwen2.5-0.5B on CUDA (bfloat16)...


/tmp/ipykernel_2736/1503701229.py:96: DeprecationWarning: HookedTransformer.from_pretrained is deprecated and will be removed in a future major release. Use TransformerBridge.boot_transformers(...) instead, then call enable_compatibility_mode() for HookedTransformer-equivalent numerics. See docs/source/content/migrating_to_v3.md.
  self.model = HookedTransformer.from_pretrained(


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Loaded pretrained model Qwen/Qwen2.5-0.5B into HookedTransformer

🔬 MILESTONE 8.1: MULTI-SEED ROBUSTNESS AUDIT (LABEL-FREE ACTIVE PROBING)
Seed   42 | L21H06:  8 | L21H01:  6 | Controls (L17/L14/L10):  4 | Active Circuit:  77.8%
Seed  137 | L21H06:  8 | L21H01:  5 | Controls (L17/L14/L10):  5 | Active Circuit:  72.2%
Seed  999 | L21H06:  9 | L21H01:  5 | Controls (L17/L14/L10):  4 | Active Circuit:  77.8%

📊 AGGREGATE METRICS ACROSS 3 INDEPENDENT SEEDS:
  • Mean Active Circuit Allocation Rate : 75.93% ± 2.62%
  • Individual Seed Rates                : ['77.8%', '72.2%', '77.8%']
--------------------------------------------------------------------------------------------------------------
👉 MULTI-SEED AUDIT VERDICT: 🟢 ROBUST_LABEL_FREE_DISCOVERY_VALIDATED
   (Active circuit prioritization is statistically stable across permutations and orthogonal bases)



In [9]:
"""
m8_2_preemptive_failure_detection.py
=====================================
Milestone 8.2: Pre-emptive Model Failure Detection via L21 Micro-Probing.

Core Objective:
  - Detect impending base-model errors on adversarial distractor prompts
    using ONLY unsupervised L21 probe responses (No Ground-Truth Leakage).
  - Compute ROC-AUC, Precision, Recall, and F1-Score for pre-emptive error detection.
"""

import torch
import numpy as np
import json
from typing import Dict, List, Tuple
from transformer_lens import HookedTransformer

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)


# ==============================================================================
# 1. MECHANISTIC PROBING & DETECTION ENGINE
# ==============================================================================

class FailureDetectionEngine:
    def __init__(self, model_name: str = "Qwen/Qwen2.5-0.5B", device: str = "cuda"):
        self.device = device if torch.cuda.is_available() else "cpu"
        print(f"🔄 Loading {model_name} on {self.device.upper()} (bfloat16)...")
        self.model = HookedTransformer.from_pretrained(
            model_name, device=self.device, torch_dtype=torch.bfloat16, default_padding_side="left"
        )
        self.model.eval()
        self.target_layer = 21
        self.target_head = 6  # Primary decision head

    @torch.no_grad()
    def evaluate_task(self, item: Dict) -> Dict:
        prompt = item["prompt"]
        tokens = self.model.to_tokens(prompt)

        # 1. Nominal Forward Pass
        nom_logits = self.model(tokens)
        last_logits = nom_logits[0, -1, :]
        top_tokens = torch.topk(last_logits, k=2).indices
        t1, t2 = int(top_tokens[0].item()), int(top_tokens[1].item())

        nom_margin = float((last_logits[t1] - last_logits[t2]).item())
        pred_token_str = self.model.to_string(t1)
        correct_token_str = item["correct"]

        # Ground-truth error flag (for audit only, not exposed to detector)
        c_id = int(self.model.to_single_token(correct_token_str))
        is_actual_error = int(t1 != c_id)

        # 2. Execute Micro-Probes on L21H06 (P1: Scale-Down 0.90, P2: Scale-Up 1.10)
        # Probe P1 (ScaleDown)
        def p1_hook(act, hook):
            act[:, -1, self.target_head, :] *= 0.90
            return act
        with self.model.hooks(fwd_hooks=[(f"blocks.{self.target_layer}.attn.hook_z", p1_hook)]):
            p1_logits = self.model(tokens)
        p1_margin = float((p1_logits[0, -1, t1] - p1_logits[0, -1, t2]).item())
        resp_p1 = nom_margin - p1_margin

        # Probe P2 (ScaleUp)
        def p2_hook(act, hook):
            act[:, -1, self.target_head, :] *= 1.10
            return act
        with self.model.hooks(fwd_hooks=[(f"blocks.{self.target_layer}.attn.hook_z", p2_hook)]):
            p2_logits = self.model(tokens)
        p2_margin = float((p2_logits[0, -1, t1] - p2_logits[0, -1, t2]).item())
        resp_p2 = nom_margin - p2_margin

        # 3. Compute Unsupervised Error Risk Score
        # High probe sensitivity / margin fragility indicates conflicting or adversarial state
        instability = abs(resp_p1) + abs(resp_p2)
        margin_penalty = float(1.0 / (1.0 + max(0.0, nom_margin)))

        # Combined Risk Score (0.0 to ~3.0+)
        error_risk_score = float(1.8 * instability + 1.2 * margin_penalty)

        return {
            "id": item["id"],
            "type": item["type"],
            "prompt": prompt,
            "pred_token": pred_token_str.strip(),
            "correct_token": correct_token_str.strip(),
            "is_actual_error": is_actual_error,
            "nom_margin": nom_margin,
            "resp_p1": resp_p1,
            "resp_p2": resp_p2,
            "risk_score": error_risk_score
        }


# ==============================================================================
# 2. EVALUATION DATASET (CLEAN vs ADVERSARIAL DISTRACTORS)
# ==============================================================================

def get_detection_benchmark():
    return [
        # --- 10 Clean Tasks (Standard Facts / In-Context Bindings) ---
        {"id": "CL01_Paris", "type": "CLEAN", "prompt": "The capital of France is the city of", "correct": " Paris"},
        {"id": "CL02_Tokyo", "type": "CLEAN", "prompt": "The capital of Japan is the bustling city of", "correct": " Tokyo"},
        {"id": "CL03_Berlin", "type": "CLEAN", "prompt": "The official capital city of Germany is", "correct": " Berlin"},
        {"id": "CL04_Madrid", "type": "CLEAN", "prompt": "The city serving as the capital of Spain is definitely", "correct": " Madrid"},
        {"id": "CL05_Rome", "type": "CLEAN", "prompt": "The historic capital of Italy is the city of", "correct": " Rome"},
        {"id": "CL06_London", "type": "CLEAN", "prompt": "The famous Big Ben clock tower is located in", "correct": " London"},
        {"id": "CL07_BindingBob", "type": "CLEAN", "prompt": "Alice handed a red apple to Bob. The apple was received by", "correct": " Bob"},
        {"id": "CL08_BindingSarah", "type": "CLEAN", "prompt": "David passed the heavy book to Sarah. The book was taken by", "correct": " Sarah"},
        {"id": "CL09_Moscow", "type": "CLEAN", "prompt": "The historic Kremlin is located in the city of", "correct": " Moscow"},
        {"id": "CL10_China", "type": "CLEAN", "prompt": "The Great Wall is a world landmark located in", "correct": " China"},

        # --- 10 Adversarial Distractor Tasks (High Interference / Deceptive Clauses) ---
        {"id": "ADV01_ColosseumParis", "type": "ADV", "prompt": "While Rome has the Colosseum, the Eiffel Tower stands in", "correct": " Paris"},
        {"id": "ADV02_MadridBerlin", "type": "ADV", "prompt": "Even though Madrid is famous, the capital of Germany is", "correct": " Berlin"},
        {"id": "ADV03_BerlinRome", "type": "ADV", "prompt": "Berlin has monuments, yet the ancient Colosseum stands in", "correct": " Rome"},
        {"id": "ADV04_TokyoAthens", "type": "ADV", "prompt": "Although Tokyo is very large, the ancient Parthenon is located in", "correct": " Athens"},
        {"id": "ADV05_LondonMadrid", "type": "ADV", "prompt": "Despite London being historic, the capital of Spain is definitely", "correct": " Madrid"},
        {"id": "ADV06_ParisRome", "type": "ADV", "prompt": "Paris is beautiful, but the capital of Italy is definitely", "correct": " Rome"},
        {"id": "ADV07_NotRomeBerlin", "type": "ADV", "prompt": "The official capital of Germany is definitely not Rome, it is", "correct": " Berlin"},
        {"id": "ADV08_EgyptParis", "type": "ADV", "prompt": "Egypt has pyramids, but the Eiffel Tower was built in the city of", "correct": " Paris"},
        {"id": "ADV09_BindingInterfere", "type": "ADV", "prompt": "Alice gave a book to Bob, but Charlie immediately seized it. The book is with", "correct": " Charlie"},
        {"id": "ADV10_MoscowParis", "type": "ADV", "prompt": "Although Moscow has the Kremlin, the capital of France is the city of", "correct": " Paris"}
    ]


# ==============================================================================
# 3. METRIC COMPUTATION ENGINE
# ==============================================================================

def compute_classification_metrics(y_true: List[int], scores: List[float]):
    """Computes ROC-AUC, Best-F1, Precision, and Recall using pure NumPy."""
    y_true = np.array(y_true)
    scores = np.array(scores)

    # ROC-AUC via rank sum (Mann-Whitney U statistic)
    pos = scores[y_true == 1]
    neg = scores[y_true == 0]

    if len(pos) == 0 or len(neg) == 0:
        auc = 0.50
    else:
        ranks = np.argsort(np.argsort(scores))
        u_stat = np.sum(ranks[y_true == 1]) - len(pos) * (len(pos) + 1) / 2.0
        auc = float(u_stat / (len(pos) * len(neg)))

    # Find optimal F1 threshold
    thresholds = np.unique(scores)
    best_f1, best_p, best_r, best_th = 0.0, 0.0, 0.0, 0.0

    for th in thresholds:
        preds = (scores >= th).astype(int)
        tp = np.sum((preds == 1) & (y_true == 1))
        fp = np.sum((preds == 1) & (y_true == 0))
        fn = np.sum((preds == 0) & (y_true == 1))

        p = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        r = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        f1 = (2 * p * r) / (p + r) if (p + r) > 0 else 0.0

        if f1 > best_f1:
            best_f1, best_p, best_r, best_th = f1, p, r, th

    return {
        "roc_auc": auc,
        "best_f1": best_f1,
        "precision": best_p,
        "recall": best_r,
        "optimal_threshold": float(best_th),
        "n_pos": int(len(pos)),
        "n_neg": int(len(neg))
    }


# ==============================================================================
# 4. MASTER RUNNER
# ==============================================================================

def run_milestone_8_2():
    engine = FailureDetectionEngine()
    benchmark = get_detection_benchmark()

    # Token integrity check
    for item in benchmark:
        _ = engine.model.to_single_token(item["correct"])
    print(f"✅ Verified single-token integrity across {len(benchmark)} benchmark tasks.")

    print("\n" + "=" * 130)
    print("🔬 MILESTONE 8.2: PRE-EMPTIVE FAILURE DETECTION BENCHMARK (L21H06 ACTIVE PROBING)")
    print("=" * 130)
    print(f"{'ID':<20} | {'Type':<6} | {'Pred Token':<12} | {'Correct':<10} | {'Status':<10} | {'Nom Margin':<11} | {'Risk Score':<12} | {'Resp P1/P2'}")
    print("-" * 130)

    results = []
    y_true = []
    risk_scores = []

    for item in benchmark:
        res = engine.evaluate_task(item)
        results.append(res)
        y_true.append(res["is_actual_error"])
        risk_scores.append(res["risk_score"])

        status_str = "❌ ERROR" if res["is_actual_error"] else "✅ CORRECT"
        probe_str = f"{res['resp_p1']:+.2f} / {res['resp_p2']:+.2f}"

        print(f"{res['id']:<20} | {res['type']:<6} | {res['pred_token']:<12} | {res['correct_token']:<10} | {status_str:<10} | {res['nom_margin']:9.3f}   | {res['risk_score']:10.4f}   | {probe_str}")

    # -------------------------------------------------------------------------
    # STATISTICAL & CLASSIFICATION AUDIT
    # -------------------------------------------------------------------------
    metrics = compute_classification_metrics(y_true, risk_scores)

    print("\n" + "=" * 130)
    print("📊 PRE-EMPTIVE ERROR DETECTION PERFORMANCE METRICS")
    print("=" * 130)
    print(f"  • Actual Model Errors in Benchmark : {metrics['n_pos']} / {len(benchmark)} ({metrics['n_pos']/len(benchmark)*100:.1f}%)")
    print(f"  • Area Under ROC Curve (ROC-AUC)   : {metrics['roc_auc']:.4f}")
    print(f"  • Optimal F1-Score                 : {metrics['best_f1']:.4f} (Threshold = {metrics['optimal_threshold']:.4f})")
    print(f"  • Precision at Optimal Threshold   : {metrics['precision']:.4f}")
    print(f"  • Recall at Optimal Threshold      : {metrics['recall']:.4f}")
    print("-" * 130)

    if metrics["roc_auc"] >= 0.80 and metrics["best_f1"] >= 0.70:
        print("👉 AUDIT VERDICT: 🟢 PRE_EMPTIVE_FAILURE_DETECTION_VALIDATED")
        print("   (Active micro-probing reliably flags impending model errors prior to output generation)")
    else:
        print("👉 AUDIT VERDICT: 🔴 INSUFFICIENT_DETECTION_ACCURACY")
    print("=" * 130 + "\n")

    with open("m8_2_failure_detection_results.json", "w", encoding="utf-8") as f:
        json.dump({"metrics": metrics, "task_traces": results}, f, indent=2, ensure_ascii=False)
    print("📁 Telemetry saved to: m8_2_failure_detection_results.json")


if __name__ == "__main__":
    run_milestone_8_2()

🔄 Loading Qwen/Qwen2.5-0.5B on CUDA (bfloat16)...


/tmp/ipykernel_2736/2640332509.py:31: DeprecationWarning: HookedTransformer.from_pretrained is deprecated and will be removed in a future major release. Use TransformerBridge.boot_transformers(...) instead, then call enable_compatibility_mode() for HookedTransformer-equivalent numerics. See docs/source/content/migrating_to_v3.md.
  self.model = HookedTransformer.from_pretrained(


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Loaded pretrained model Qwen/Qwen2.5-0.5B into HookedTransformer
✅ Verified single-token integrity across 20 benchmark tasks.

🔬 MILESTONE 8.2: PRE-EMPTIVE FAILURE DETECTION BENCHMARK (L21H06 ACTIVE PROBING)
ID                   | Type   | Pred Token   | Correct    | Status     | Nom Margin  | Risk Score   | Resp P1/P2
----------------------------------------------------------------------------------------------------------------------------------
CL01_Paris           | CLEAN  | Paris        | Paris      | ✅ CORRECT  |     2.750   |     0.5450   | -0.12 / +0.00
CL02_Tokyo           | CLEAN  | Tokyo        | Tokyo      | ✅ CORRECT  |     4.125   |     0.2341   | +0.00 / +0.00
CL03_Berlin          | CLEAN  | Berlin       | Berlin     | ✅ CORRECT  |     0.250   |     1.1850   | -0.12 / +0.00
CL04_Madrid          | CLEAN  | the          | Madrid     | ❌ ERROR    |     0.250   |     0.9600   | +0.00 / +0.00
CL05_Rome            | CLEAN  | Rome         | Rome       | ✅ CORRECT  |     0.875  

In [10]:
"""
m8_3_dynamic_activation_steering.py
====================================
Milestone 8.3: Dynamic Activation Steering & Autonomous Self-Repair Benchmark.

Core Protocol:
  1. Contrastive Vector Extraction: Computes entity-promotion steering vector (v_steer)
     on a small calibration set (disjoint from evaluation tasks).
  2. Targeted Dynamic Steering: Injects v_steer into L21H06 during the forward pass.
  3. Control Specificity Audit: Applies identical-norm steering to control heads (L17H01, L14H04).
  4. Full Benchmark: Evaluates 20 tasks (10 Clean, 10 Adversarial) across 3 conditions:
     - Condition A: Base Model (No Steering)
     - Condition B: L21H06 Dynamic Steering (Target Circuit)
     - Condition C: L17H01 Dynamic Steering (Control Head)
"""

import torch
import numpy as np
import json
from typing import Dict, List, Tuple
from transformer_lens import HookedTransformer

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)


# ==============================================================================
# 1. STEERING HARNESS & CALIBRATION ENGINE
# ==============================================================================

class DynamicSteeringHarness:
    def __init__(self, model_name: str = "Qwen/Qwen2.5-0.5B", device: str = "cuda"):
        self.device = device if torch.cuda.is_available() else "cpu"
        print(f"🔄 Loading {model_name} on {self.device.upper()} (bfloat16)...")
        self.model = HookedTransformer.from_pretrained(
            model_name, device=self.device, torch_dtype=torch.bfloat16, default_padding_side="left"
        )
        self.model.eval()
        self.target_l = 21
        self.target_h = 6   # Decision head
        self.ctrl_l = 17
        self.ctrl_h = 1     # Control head
        self.d_head = self.model.cfg.d_head

        # Extract contrastive steering vector
        self.v_steer_target = self._calibrate_steering_vector(self.target_l, self.target_h)
        self.v_steer_ctrl = self._calibrate_steering_vector(self.ctrl_l, self.ctrl_h)

    def _calibrate_steering_vector(self, layer: int, head: int) -> torch.Tensor:
        """Extracts contrastive vector from 4 disjoint calibration pairs (Entity vs Syntax/Distractor)."""
        calib_pairs = [
            ("The capital city of Italy is", "The capital city of Italy is not"),
            ("The famous landmark in Paris is the", "The famous landmark in Paris is located"),
            ("The currency used in Japan is the", "The currency used in Japan is officially"),
            ("The large country containing Moscow is", "The large country containing Moscow has")
        ]

        diff_vectors = []
        for p_entity, p_syntax in calib_pairs:
            toks_e = self.model.to_tokens(p_entity)
            toks_s = self.model.to_tokens(p_syntax)

            _, cache_e = self.model.run_with_cache(toks_e)
            _, cache_s = self.model.run_with_cache(toks_s)

            z_e = cache_e[f"blocks.{layer}.attn.hook_z"][0, -1, head, :].detach().cpu().to(torch.float32)
            z_s = cache_s[f"blocks.{layer}.attn.hook_z"][0, -1, head, :].detach().cpu().to(torch.float32)

            diff_vectors.append(z_e - z_s)

        mean_diff = torch.stack(diff_vectors).mean(dim=0)
        unit_vec = mean_diff / (mean_diff.norm() + 1e-8)
        return unit_vec.to(self.device)

    @torch.no_grad()
    def evaluate_condition(self, prompt: str, correct_str: str, mode: str = "BASE", alpha: float = 2.5) -> Dict:
        tokens = self.model.to_tokens(prompt)
        c_id = int(self.model.to_single_token(correct_str))

        if mode == "BASE":
            logits = self.model(tokens)
        elif mode == "TARGET_L21H06":
            def steer_hook(act, hook):
                v = self.v_steer_target.to(act.dtype)
                act[:, -1, self.target_h, :] += alpha * v
                return act
            with self.model.hooks(fwd_hooks=[(f"blocks.{self.target_l}.attn.hook_z", steer_hook)]):
                logits = self.model(tokens)
        elif mode == "CTRL_L17H01":
            def ctrl_hook(act, hook):
                v = self.v_steer_ctrl.to(act.dtype)
                act[:, -1, self.ctrl_h, :] += alpha * v
                return act
            with self.model.hooks(fwd_hooks=[(f"blocks.{self.ctrl_l}.attn.hook_z", ctrl_hook)]):
                logits = self.model(tokens)
        else:
            raise ValueError(f"Unknown mode: {mode}")

        last_logits = logits[0, -1, :]
        top_id = int(torch.argmax(last_logits).item())
        pred_str = self.model.to_string(top_id).strip()
        is_correct = (top_id == c_id)

        # Logit margin of correct token over top alternative
        alt_id = int(torch.topk(last_logits, k=2).indices[1].item()) if top_id == c_id else top_id
        margin = float((last_logits[c_id] - last_logits[alt_id]).item())

        return {
            "pred_str": pred_str,
            "is_correct": is_correct,
            "margin": margin
        }


# ==============================================================================
# 2. BENCHMARK CORPUS (20 TASKS: 10 CLEAN + 10 ADVERSARIAL)
# ==============================================================================

def get_full_benchmark():
    return [
        # --- 10 Clean Tasks ---
        {"id": "CL01_Paris", "type": "CLEAN", "prompt": "The capital of France is the city of", "correct": " Paris"},
        {"id": "CL02_Tokyo", "type": "CLEAN", "prompt": "The capital of Japan is the bustling city of", "correct": " Tokyo"},
        {"id": "CL03_Berlin", "type": "CLEAN", "prompt": "The official capital city of Germany is", "correct": " Berlin"},
        {"id": "CL04_Madrid", "type": "CLEAN", "prompt": "The city serving as the capital of Spain is definitely", "correct": " Madrid"},
        {"id": "CL05_Rome", "type": "CLEAN", "prompt": "The historic capital of Italy is the city of", "correct": " Rome"},
        {"id": "CL06_London", "type": "CLEAN", "prompt": "The famous Big Ben clock tower is located in", "correct": " London"},
        {"id": "CL07_BindingBob", "type": "CLEAN", "prompt": "Alice handed a red apple to Bob. The apple was received by", "correct": " Bob"},
        {"id": "CL08_BindingSarah", "type": "CLEAN", "prompt": "David passed the heavy book to Sarah. The book was taken by", "correct": " Sarah"},
        {"id": "CL09_Moscow", "type": "CLEAN", "prompt": "The historic Kremlin is located in the city of", "correct": " Moscow"},
        {"id": "CL10_China", "type": "CLEAN", "prompt": "The Great Wall is a world landmark located in", "correct": " China"},

        # --- 10 Adversarial Distractor Tasks ---
        {"id": "ADV01_ColosseumParis", "type": "ADV", "prompt": "While Rome has the Colosseum, the Eiffel Tower stands in", "correct": " Paris"},
        {"id": "ADV02_MadridBerlin", "type": "ADV", "prompt": "Even though Madrid is famous, the capital of Germany is", "correct": " Berlin"},
        {"id": "ADV03_BerlinRome", "type": "ADV", "prompt": "Berlin has monuments, yet the ancient Colosseum stands in", "correct": " Rome"},
        {"id": "ADV04_TokyoAthens", "type": "ADV", "prompt": "Although Tokyo is very large, the ancient Parthenon is located in", "correct": " Athens"},
        {"id": "ADV05_LondonMadrid", "type": "ADV", "prompt": "Despite London being historic, the capital of Spain is definitely", "correct": " Madrid"},
        {"id": "ADV06_ParisRome", "type": "ADV", "prompt": "Paris is beautiful, but the capital of Italy is definitely", "correct": " Rome"},
        {"id": "ADV07_NotRomeBerlin", "type": "ADV", "prompt": "The official capital of Germany is definitely not Rome, it is", "correct": " Berlin"},
        {"id": "ADV08_EgyptParis", "type": "ADV", "prompt": "Egypt has pyramids, but the Eiffel Tower was built in the city of", "correct": " Paris"},
        {"id": "ADV09_BindingInterfere", "type": "ADV", "prompt": "Alice gave a book to Bob, but Charlie immediately seized it. The book is with", "correct": " Charlie"},
        {"id": "ADV10_MoscowParis", "type": "ADV", "prompt": "Although Moscow has the Kremlin, the capital of France is the city of", "correct": " Paris"}
    ]


# ==============================================================================
# 3. BENCHMARK EXECUTION & AUDIT
# ==============================================================================

def run_milestone_8_3():
    harness = DynamicSteeringHarness()
    benchmark = get_full_benchmark()

    # Verify single-token integrity
    for item in benchmark:
        _ = harness.model.to_single_token(item["correct"])
    print(f"✅ Verified single-token integrity across {len(benchmark)} evaluation tasks.")

    print("\n" + "=" * 140)
    print("🔬 MILESTONE 8.3: DYNAMIC ACTIVATION STEERING & SELF-REPAIR BENCHMARK (α = 2.5)")
    print("=" * 140)
    print(f"{'ID':<20} | {'Type':<6} | {'Target':<9} | {'Base Output':<14} | {'L21H06 Steered':<16} | {'L17H01 Control':<16} | {'Self-Repair Status'}")
    print("-" * 140)

    records = []
    base_clean_correct, steer_clean_correct, ctrl_clean_correct = 0, 0, 0
    base_adv_correct, steer_adv_correct, ctrl_adv_correct = 0, 0, 0
    adv_recovered_count, adv_broken_count = 0, 0

    for item in benchmark:
        p = item["prompt"]
        c = item["correct"]

        res_base = harness.evaluate_condition(p, c, mode="BASE")
        res_steer = harness.evaluate_condition(p, c, mode="TARGET_L21H06", alpha=2.5)
        res_ctrl = harness.evaluate_condition(p, c, mode="CTRL_L17H01", alpha=2.5)

        # Track accuracy
        if item["type"] == "CLEAN":
            if res_base["is_correct"]: base_clean_correct += 1
            if res_steer["is_correct"]: steer_clean_correct += 1
            if res_ctrl["is_correct"]: ctrl_clean_correct += 1
        else:
            if res_base["is_correct"]: base_adv_correct += 1
            if res_steer["is_correct"]: steer_adv_correct += 1
            if res_ctrl["is_correct"]: ctrl_adv_correct += 1

        # Evaluate repair transition
        if not res_base["is_correct"] and res_steer["is_correct"]:
            status_str = "🟢 REPAIRED (Recovered)"
            if item["type"] == "ADV": adv_recovered_count += 1
        elif res_base["is_correct"] and not res_steer["is_correct"]:
            status_str = "🔴 REGRESSED (Broken)"
            if item["type"] == "ADV": adv_broken_count += 1
        elif res_base["is_correct"] and res_steer["is_correct"]:
            status_str = "✅ PRESERVED (Correct)"
        else:
            status_str = "❌ PERSISTENT_ERROR"

        base_str = f"{res_base['pred_str']} ({'✓' if res_base['is_correct'] else '✗'})"
        steer_str = f"{res_steer['pred_str']} ({'✓' if res_steer['is_correct'] else '✗'})"
        ctrl_str = f"{res_ctrl['pred_str']} ({'✓' if res_ctrl['is_correct'] else '✗'})"

        print(f"{item['id']:<20} | {item['type']:<6} | {c.strip():<9} | {base_str:<14} | {steer_str:<16} | {ctrl_str:<16} | {status_str}")

        records.append({
            "id": item["id"],
            "type": item["type"],
            "correct": c.strip(),
            "base": res_base,
            "steered_l21h6": res_steer,
            "control_l17h1": res_ctrl,
            "status": status_str
        })

    # -------------------------------------------------------------------------
    # STATISTICAL REPAIR AUDIT & METRICS
    # -------------------------------------------------------------------------
    total_clean = 10
    total_adv = 10
    total_adv_base_errors = total_adv - base_adv_correct

    recovery_rate = (adv_recovered_count / total_adv_base_errors * 100.0) if total_adv_base_errors > 0 else 0.0
    clean_preservation = (steer_clean_correct / total_clean * 100.0)

    adv_acc_base = (base_adv_correct / total_adv) * 100.0
    adv_acc_steer = (steer_adv_correct / total_adv) * 100.0
    adv_acc_ctrl = (ctrl_adv_correct / total_adv) * 100.0

    print("\n" + "=" * 140)
    print("📊 MILESTONE 8.3 SUMMARY & AUDIT METRICS")
    print("=" * 140)
    print(f"  • Clean Tasks Accuracy (Base Model)       : {base_clean_correct}/{total_clean} ({(base_clean_correct/total_clean)*100:.1f}%)")
    print(f"  • Clean Tasks Accuracy (L21H06 Steered)   : {steer_clean_correct}/{total_clean} ({clean_preservation:.1f}%)")
    print(f"  • Clean Tasks Accuracy (L17H01 Control)   : {ctrl_clean_correct}/{total_clean} ({(ctrl_clean_correct/total_clean)*100:.1f}%)")
    print("-" * 140)
    print(f"  • Adversarial Tasks Accuracy (Base Model) : {base_adv_correct}/{total_adv} ({adv_acc_base:.1f}%)")
    print(f"  • Adversarial Tasks Accuracy (L21H06 Steer): {steer_adv_correct}/{total_adv} ({adv_acc_steer:.1f}%)  [Net Gain: +{adv_acc_steer - adv_acc_base:.1f}%]")
    print(f"  • Adversarial Tasks Accuracy (Control)    : {ctrl_adv_correct}/{total_adv} ({adv_acc_ctrl:.1f}%)  [Net Gain: +{adv_acc_ctrl - adv_acc_base:.1f}%]")
    print("-" * 140)
    print(f"  • Baseline Adversarial Errors Recovered   : {adv_recovered_count}/{total_adv_base_errors} ({recovery_rate:.1f}%)")
    print(f"  • Circuit Specificity Advantage (L21 vs Ctrl): +{adv_acc_steer - adv_acc_ctrl:.1f}%")
    print("=" * 140)

    # Formal Falsifiable Verdict
    if recovery_rate >= 40.0 and clean_preservation >= 60.0 and (adv_acc_steer - adv_acc_ctrl) >= 20.0:
        print("👉 FINAL AUDIT VERDICT: 🟢 DYNAMIC_STEERING_SELF_REPAIR_VALIDATED")
        print("   (Targeted L21H06 activation steering successfully repairs syntax-collapse errors with circuit specificity)")
    else:
        print("👉 FINAL AUDIT VERDICT: 🔴 INSUFFICIENT_STEERING_SPECIFICITY")
    print("=" * 140 + "\n")

    with open("m8_3_steering_results.json", "w", encoding="utf-8") as f:
        json.dump({
            "metrics": {
                "recovery_rate_pct": recovery_rate,
                "clean_preservation_pct": clean_preservation,
                "adv_acc_base_pct": adv_acc_base,
                "adv_acc_steer_pct": adv_acc_steer,
                "adv_acc_ctrl_pct": adv_acc_ctrl
            },
            "records": records
        }, f, indent=2, ensure_ascii=False)
    print("📁 Telemetry saved to: m8_3_steering_results.json")


if __name__ == "__main__":
    run_milestone_8_3()

🔄 Loading Qwen/Qwen2.5-0.5B on CUDA (bfloat16)...


/tmp/ipykernel_2736/3320333021.py:36: DeprecationWarning: HookedTransformer.from_pretrained is deprecated and will be removed in a future major release. Use TransformerBridge.boot_transformers(...) instead, then call enable_compatibility_mode() for HookedTransformer-equivalent numerics. See docs/source/content/migrating_to_v3.md.
  self.model = HookedTransformer.from_pretrained(


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Loaded pretrained model Qwen/Qwen2.5-0.5B into HookedTransformer
✅ Verified single-token integrity across 20 evaluation tasks.

🔬 MILESTONE 8.3: DYNAMIC ACTIVATION STEERING & SELF-REPAIR BENCHMARK (α = 2.5)
ID                   | Type   | Target    | Base Output    | L21H06 Steered   | L17H01 Control   | Self-Repair Status
--------------------------------------------------------------------------------------------------------------------------------------------
CL01_Paris           | CLEAN  | Paris     | Paris (✓)      | Paris (✓)        | Paris (✓)        | ✅ PRESERVED (Correct)
CL02_Tokyo           | CLEAN  | Tokyo     | Tokyo (✓)      | Tokyo (✓)        | Tokyo (✓)        | ✅ PRESERVED (Correct)
CL03_Berlin          | CLEAN  | Berlin    | Berlin (✓)     | Berlin (✓)       | Berlin (✓)       | ✅ PRESERVED (Correct)
CL04_Madrid          | CLEAN  | Madrid    | the (✗)        | the (✗)          | the (✗)          | ❌ PERSISTENT_ERROR
CL05_Rome            | CLEAN  | Rome      | Rome (✓) 